# Typed instrument API catalog

Every instrument under `finstack_quant.valuations.instruments.*` is a **typed class** sharing one surface:

- `Cls.from_json(json_str)` — build from the tagged `{"type": ..., "spec": {...}}` form
- `inst.validate()` — structural validation (raises `ValueError`)
- `inst.to_json()` — canonical round-trip
- `inst.price(market, as_of, model="default")` / `inst.price_with_metrics(...)` — pricing (needs a `MarketContext`)

This notebook round-trips a verified spec for 70 instrument classes across `commodity`, `credit_derivatives`, `equity`, `exotics`, `fixed_income`, `fx`, `rates`, then prices all of them end-to-end against one synthesized shared market on a best-effort basis.

## Instrument catalog: `from_json` -> `validate` -> `to_json`

`CATALOG` maps each module to `{ClassName: inner_spec}`. We import every class, rebuild it from its spec, validate, and confirm the JSON round-trips.

In [10]:
import json
import importlib

# Verified instrument specs (tagged inner form: {"type": ..., "spec": {...}}),
# sourced from the canonical schema examples / golden fixtures / notebooks.
CATALOG = json.loads(r'''
{
  "commodity": {
    "CommodityAsianOption": {
      "spec": {
        "attributes": {},
        "averaging_method": "arithmetic",
        "commodity_type": "Energy",
        "currency": "USD",
        "day_count": "Act365F",
        "discount_curve_id": "USD-OIS",
        "expiry": "2025-07-02",
        "fixing_dates": [
          "2025-01-31",
          "2025-02-28",
          "2025-03-31",
          "2025-04-30",
          "2025-05-31",
          "2025-06-30"
        ],
        "forward_curve_id": "CL-FORWARD",
        "id": "WTI-ASIAN-6M",
        "option_type": "call",
        "pricing_overrides": {
          "adaptive_bumps": false,
          "call_friction_cents": null,
          "cds_quote_bp": null,
          "credit_spread_bump_bp": null,
          "implied_volatility": null,
          "mc_seed_scenario": null,
          "mean_reversion": null,
          "quoted_clean_price": null,
          "rate_bump_bp": null,
          "rho_bump_decimal": null,
          "spot_bump_pct": null,
          "theta_period": null,
          "tree_steps": null,
          "upfront_payment": null,
          "use_gobet_miri": false,
          "vega_bump_decimal": null,
          "vol_bump_pct": null,
          "vol_surface_extrapolation": "error",
          "ytm_bump_decimal": null
        },
        "quantity": 1000.0,
        "realized_fixings": [],
        "strike": 75.0,
        "ticker": "CL",
        "unit": "BBL",
        "vol_surface_id": "CL-VOL"
      },
      "type": "commodity_asian_option"
    },
    "CommodityForward": {
      "spec": {
        "attributes": {
          "meta": {
            "sector": "crude"
          },
          "tags": [
            "energy"
          ]
        },
        "commodity_type": "Energy",
        "contract_month": "2025M03",
        "convention": "w_t_i_crude",
        "currency": "USD",
        "discount_curve_id": "USD-OIS",
        "exchange": "NYMEX",
        "forward_curve_id": "WTI-FORWARD",
        "id": "WTI-FWD-2025M03",
        "maturity": "2025-03-15",
        "multiplier": 1.0,
        "position": "long",
        "pricing_overrides": {
          "adaptive_bumps": false,
          "call_friction_cents": null,
          "cds_quote_bp": null,
          "credit_spread_bump_bp": null,
          "implied_volatility": null,
          "mc_seed_scenario": null,
          "mean_reversion": null,
          "quoted_clean_price": null,
          "rate_bump_bp": null,
          "rho_bump_decimal": null,
          "spot_bump_pct": null,
          "theta_period": null,
          "tree_steps": null,
          "upfront_payment": null,
          "use_gobet_miri": false,
          "vega_bump_decimal": null,
          "vol_bump_pct": null,
          "vol_surface_extrapolation": "error",
          "ytm_bump_decimal": null
        },
        "quantity": 1000.0,
        "settlement": "cash",
        "ticker": "CL",
        "unit": "BBL"
      },
      "type": "commodity_forward"
    },
    "CommodityOption": {
      "spec": {
        "attributes": {},
        "commodity_type": "Energy",
        "currency": "USD",
        "day_count": "Act365F",
        "discount_curve_id": "USD-OIS",
        "exercise_style": "european",
        "expiry": "2025-06-15",
        "forward_curve_id": "WTI-FORWARD",
        "id": "WTI-OPT-2025M06",
        "multiplier": 1.0,
        "option_type": "call",
        "pricing_overrides": {
          "adaptive_bumps": false,
          "call_friction_cents": null,
          "cds_quote_bp": null,
          "credit_spread_bump_bp": null,
          "implied_volatility": null,
          "mc_seed_scenario": null,
          "mean_reversion": null,
          "quoted_clean_price": null,
          "rate_bump_bp": null,
          "rho_bump_decimal": null,
          "spot_bump_pct": null,
          "theta_period": null,
          "tree_steps": null,
          "upfront_payment": null,
          "use_gobet_miri": false,
          "vega_bump_decimal": null,
          "vol_bump_pct": null,
          "vol_surface_extrapolation": "error",
          "ytm_bump_decimal": null
        },
        "quantity": 1000.0,
        "settlement": "cash",
        "strike": 75.0,
        "ticker": "CL",
        "unit": "BBL",
        "vol_surface_id": "WTI-VOL"
      },
      "type": "commodity_option"
    },
    "CommoditySpreadOption": {
      "spec": {
        "attributes": {},
        "correlation": 0.85,
        "currency": "USD",
        "day_count": "Act365F",
        "discount_curve_id": "USD-OIS",
        "expiry": "2025-09-15",
        "id": "WTI-RBOB-CRACK-SPREAD",
        "leg1_forward_curve_id": "RBOB-FORWARD",
        "leg1_vol_surface_id": "RBOB-VOL",
        "leg2_forward_curve_id": "WTI-FORWARD",
        "leg2_vol_surface_id": "WTI-VOL",
        "notional": 10000.0,
        "option_type": "call",
        "pricing_overrides": {
          "adaptive_bumps": false,
          "call_friction_cents": null,
          "cds_quote_bp": null,
          "credit_spread_bump_bp": null,
          "implied_volatility": null,
          "mc_seed_scenario": null,
          "mean_reversion": null,
          "quoted_clean_price": null,
          "rate_bump_bp": null,
          "rho_bump_decimal": null,
          "spot_bump_pct": null,
          "theta_period": null,
          "tree_steps": null,
          "upfront_payment": null,
          "use_gobet_miri": false,
          "vega_bump_decimal": null,
          "vol_bump_pct": null,
          "vol_surface_extrapolation": "error",
          "ytm_bump_decimal": null
        },
        "strike": 5.0
      },
      "type": "commodity_spread_option"
    },
    "CommoditySwap": {
      "spec": {
        "attributes": {
          "meta": {
            "sector": "natural-gas"
          },
          "tags": [
            "energy"
          ]
        },
        "bdc": "modified_following",
        "commodity_type": "Energy",
        "currency": "USD",
        "discount_curve_id": "USD-OIS",
        "fixed_price": "3.5",
        "floating_index_id": "NG-SPOT-AVG",
        "frequency": {
          "count": 1,
          "unit": "months"
        },
        "id": "NG-SWAP-2025",
        "maturity": "2025-12-31",
        "pricing_overrides": {
          "adaptive_bumps": false,
          "call_friction_cents": null,
          "cds_quote_bp": null,
          "credit_spread_bump_bp": null,
          "implied_volatility": null,
          "mc_seed_scenario": null,
          "mean_reversion": null,
          "quoted_clean_price": null,
          "rate_bump_bp": null,
          "rho_bump_decimal": null,
          "spot_bump_pct": null,
          "theta_period": null,
          "tree_steps": null,
          "upfront_payment": null,
          "use_gobet_miri": false,
          "vega_bump_decimal": null,
          "vol_bump_pct": null,
          "vol_surface_extrapolation": "error",
          "ytm_bump_decimal": null
        },
        "quantity": 10000.0,
        "side": "pay",
        "start_date": "2025-01-01",
        "ticker": "NG",
        "unit": "MMBTU"
      },
      "type": "commodity_swap"
    },
    "CommoditySwaption": {
      "spec": {
        "attributes": {},
        "bdc": "modified_following",
        "commodity_type": "Energy",
        "currency": "USD",
        "day_count": "Act365F",
        "discount_curve_id": "USD-OIS",
        "expiry": "2025-06-15",
        "fixed_price": 3.5,
        "forward_curve_id": "NG-FORWARD",
        "id": "NG-SWAPTION-2025",
        "notional": 10000.0,
        "option_type": "call",
        "pricing_overrides": {
          "adaptive_bumps": false,
          "call_friction_cents": null,
          "cds_quote_bp": null,
          "credit_spread_bump_bp": null,
          "implied_volatility": null,
          "mc_seed_scenario": null,
          "mean_reversion": null,
          "quoted_clean_price": null,
          "rate_bump_bp": null,
          "rho_bump_decimal": null,
          "spot_bump_pct": null,
          "theta_period": null,
          "tree_steps": null,
          "upfront_payment": null,
          "use_gobet_miri": false,
          "vega_bump_decimal": null,
          "vol_bump_pct": null,
          "vol_surface_extrapolation": "error",
          "ytm_bump_decimal": null
        },
        "swap_end": "2026-06-30",
        "swap_frequency": {
          "count": 1,
          "unit": "months"
        },
        "swap_start": "2025-07-01",
        "ticker": "NG",
        "unit": "MMBTU",
        "vol_surface_id": "NG-VOL"
      },
      "type": "commodity_swaption"
    }
  },
  "credit_derivatives": {
    "CDSIndex": {
      "spec": {
        "attributes": {
          "meta": {},
          "tags": []
        },
        "constituents": [],
        "convention": "isda_na",
        "id": "CDX-IG-42",
        "index_factor": 1.0,
        "index_name": "CDX.NA.IG",
        "notional": {
          "amount": "10000000",
          "currency": "USD"
        },
        "premium": {
          "bdc": "following",
          "calendar_id": null,
          "day_count": "Act360",
          "discount_curve_id": "USD-OIS",
          "end": "2029-12-20",
          "frequency": {
            "count": 3,
            "unit": "months"
          },
          "spread_bp": 60.0,
          "start": "2024-03-20",
          "stub": "ShortFront"
        },
        "pricing": "SingleCurve",
        "pricing_overrides": {
          "adaptive_bumps": false,
          "cds_quote_bp": null,
          "implied_volatility": null,
          "mc_seed_scenario": null,
          "quoted_clean_price": null,
          "rate_bump_bp": null,
          "rho_bump_decimal": null,
          "spot_bump_pct": null,
          "theta_period": null,
          "upfront_payment": null,
          "vega_bump_decimal": null,
          "vol_bump_pct": null,
          "ytm_bump_decimal": null
        },
        "protection": {
          "credit_curve_id": "CDX.NA.IG.HAZARD",
          "recovery_rate": 0.4,
          "settlement_delay": 3
        },
        "series": 42,
        "side": "pay",
        "version": 1
      },
      "type": "cds_index"
    },
    "CDSOption": {
      "spec": {
        "attributes": {
          "meta": {},
          "tags": []
        },
        "cds_maturity": "2030-06-20",
        "credit_curve_id": "CORP-HAZARD",
        "discount_curve_id": "USD-OIS",
        "exercise_style": "european",
        "expiry": "2025-06-20",
        "id": "CDSOPT-CALL-CORP-5Y",
        "index_factor": null,
        "notional": {
          "amount": "10000000",
          "currency": "USD"
        },
        "option_type": "call",
        "pricing_overrides": {
          "adaptive_bumps": false,
          "cds_quote_bp": null,
          "implied_volatility": null,
          "mc_seed_scenario": null,
          "quoted_clean_price": null,
          "rate_bump_bp": null,
          "rho_bump_decimal": null,
          "spot_bump_pct": null,
          "theta_period": null,
          "upfront_payment": null,
          "vega_bump_decimal": null,
          "vol_bump_pct": null,
          "ytm_bump_decimal": null
        },
        "realized_index_loss": null,
        "recovery_rate": 0.4,
        "settlement": "cash",
        "strike": "0.01",
        "underlying_is_index": false,
        "vol_surface_id": "CDSOPT-VOL"
      },
      "type": "cds_option"
    },
    "CDSTranche": {
      "spec": {
        "accumulated_loss": 0.0,
        "attach_pct": 0.0,
        "attributes": {
          "meta": {},
          "tags": []
        },
        "bdc": "following",
        "calendar_id": "weekends_only",
        "credit_index_id": "CDX.NA.IG.HAZARD",
        "day_count": "Act360",
        "detach_pct": 3.0,
        "discount_curve_id": "USD-OIS",
        "frequency": {
          "count": 3,
          "unit": "months"
        },
        "id": "CDX-0-3",
        "index_name": "CDX.NA.IG",
        "maturity": "2029-12-20",
        "notional": {
          "amount": "10000000",
          "currency": "USD"
        },
        "running_coupon_bp": 500.0,
        "series": 42,
        "side": "buy_protection",
        "standard_imm_dates": false
      },
      "type": "cds_tranche"
    },
    "CreditDefaultSwap": {
      "spec": {
        "attributes": {},
        "convention": "isda_na",
        "id": "CDS-CORP-5Y",
        "notional": {
          "amount": 10000000.0,
          "currency": "USD"
        },
        "premium": {
          "bdc": "following",
          "calendar_id": "usny",
          "day_count": "Act360",
          "discount_curve_id": "USD-OIS",
          "end": "2030-03-20",
          "frequency": {
            "count": 3,
            "unit": "months"
          },
          "spread_bp": 100.0,
          "start": "2025-03-20",
          "stub": "ShortFront"
        },
        "pricing_overrides": {},
        "protection": {
          "credit_curve_id": "CORP-HAZARD",
          "recovery_rate": 0.4,
          "settlement_delay": 3
        },
        "side": "pay"
      },
      "type": "credit_default_swap"
    }
  },
  "equity": {
    "Autocallable": {
      "spec": {
        "attributes": {
          "meta": {},
          "tags": []
        },
        "autocall_barriers": [
          1.0,
          1.0,
          1.0,
          1.0
        ],
        "cap_level": 1.5,
        "coupons": [
          0.02,
          0.02,
          0.02,
          0.02
        ],
        "day_count": "Act365F",
        "discount_curve_id": "USD-OIS",
        "div_yield_id": "SPX-DIV",
        "expiry": "2024-12-31",
        "final_barrier": 0.6,
        "final_payoff_type": {
          "Participation": {
            "rate": 1.0
          }
        },
        "id": "AUTO-SPX-QTR",
        "notional": {
          "amount": "100000",
          "currency": "USD"
        },
        "observation_dates": [
          "2024-03-29",
          "2024-06-28",
          "2024-09-30",
          "2024-12-31"
        ],
        "participation_rate": 1.0,
        "pricing_overrides": {
          "adaptive_bumps": false,
          "cds_quote_bp": null,
          "implied_volatility": null,
          "mc_seed_scenario": null,
          "quoted_clean_price": null,
          "rate_bump_bp": null,
          "rho_bump_decimal": null,
          "spot_bump_pct": null,
          "theta_period": null,
          "upfront_payment": null,
          "vega_bump_decimal": null,
          "vol_bump_pct": null,
          "ytm_bump_decimal": null
        },
        "spot_id": "SPX-SPOT",
        "underlying_ticker": "SPX",
        "vol_surface_id": "SPX-VOL"
      },
      "type": "autocallable"
    },
    "CliquetOption": {
      "spec": {
        "attributes": {
          "meta": {},
          "tags": []
        },
        "day_count": "Act365F",
        "discount_curve_id": "USD-OIS",
        "div_yield_id": "SPX-DIV",
        "expiry": "2024-12-31",
        "global_cap": 0.2,
        "global_floor": 0.0,
        "id": "CLIQ-SPX-QTR",
        "local_cap": 0.05,
        "local_floor": -0.05,
        "notional": {
          "amount": "100000",
          "currency": "USD"
        },
        "pricing_overrides": {
          "adaptive_bumps": false,
          "cds_quote_bp": null,
          "implied_volatility": null,
          "mc_seed_scenario": null,
          "quoted_clean_price": null,
          "rate_bump_bp": null,
          "rho_bump_decimal": null,
          "spot_bump_pct": null,
          "theta_period": null,
          "upfront_payment": null,
          "vega_bump_decimal": null,
          "vol_bump_pct": null,
          "ytm_bump_decimal": null
        },
        "reset_dates": [
          "2024-03-29",
          "2024-06-28",
          "2024-09-30",
          "2024-12-31"
        ],
        "spot_id": "SPX-SPOT",
        "underlying_ticker": "SPX",
        "vol_surface_id": "SPX-VOL"
      },
      "type": "cliquet_option"
    },
    "DiscountedCashFlow": {
      "spec": {
        "attributes": {},
        "currency": "USD",
        "discount_curve_id": "USD-OIS",
        "flows": [
          [
            "2026-01-01",
            5000000.0
          ],
          [
            "2027-01-01",
            5750000.0
          ],
          [
            "2028-01-01",
            6500000.0
          ],
          [
            "2029-01-01",
            7250000.0
          ],
          [
            "2030-01-01",
            8000000.0
          ]
        ],
        "id": "DCF-TECH-CO",
        "mid_year_convention": true,
        "net_debt": 15000000.0,
        "pricing_overrides": {
          "adaptive_bumps": false,
          "call_friction_cents": null,
          "cds_quote_bp": null,
          "credit_spread_bump_bp": null,
          "implied_volatility": null,
          "mc_seed_scenario": null,
          "mean_reversion": null,
          "quoted_clean_price": null,
          "rate_bump_bp": null,
          "rho_bump_decimal": null,
          "spot_bump_pct": null,
          "theta_period": null,
          "tree_steps": null,
          "upfront_payment": null,
          "use_gobet_miri": false,
          "vega_bump_decimal": null,
          "vol_bump_pct": null,
          "vol_surface_extrapolation": "error",
          "ytm_bump_decimal": null
        },
        "shares_outstanding": 10000000.0,
        "terminal_value": {
          "growth_rate": 0.02,
          "type": "gordon_growth"
        },
        "valuation_date": "2025-01-01",
        "wacc": 0.1
      },
      "type": "discounted_cash_flow"
    },
    "Equity": {
      "spec": {
        "attributes": {
          "meta": {},
          "tags": []
        },
        "currency": "USD",
        "discount_curve_id": "USD",
        "div_yield_id": "AAPL-DIV",
        "id": "EQUITY-AAPL",
        "price_id": "AAPL-SPOT",
        "price_quote": null,
        "shares": 100.0,
        "ticker": "AAPL"
      },
      "type": "equity"
    },
    "EquityIndexFuture": {
      "spec": {
        "attributes": {
          "meta": {},
          "tags": [
            "golden"
          ]
        },
        "contract_specs": {
          "multiplier": 50.0,
          "settlement_method": "cash",
          "tick_size": 0.25,
          "tick_value": 12.5
        },
        "discount_curve_id": "USD-OIS",
        "div_yield_id": "SPX-DIVYIELD",
        "entry_price": 4500.0,
        "expiry": "2026-07-30",
        "id": "SPX-ES-3M-GOLDEN",
        "last_trading_date": "2026-07-29",
        "notional": {
          "amount": "2250000",
          "currency": "USD"
        },
        "position": "long",
        "spot_id": "SPX-SPOT",
        "underlying_ticker": "SPX"
      },
      "type": "equity_index_future"
    },
    "EquityOption": {
      "spec": {
        "attributes": {},
        "day_count": "Act365F",
        "discount_curve_id": "USD-OIS",
        "div_yield_id": "EQUITY-DIVYIELD",
        "exercise_style": "european",
        "expiry": "2026-06-15",
        "id": "SPX-CALL-4500",
        "notional": {
          "amount": 100.0,
          "currency": "USD"
        },
        "option_type": "call",
        "pricing_overrides": {},
        "settlement": "cash",
        "spot_id": "EQUITY-SPOT",
        "strike": 4500.0,
        "underlying_ticker": "SPX",
        "vol_surface_id": "EQUITY-VOL"
      },
      "type": "equity_option"
    },
    "EquityTotalReturnSwap": {
      "spec": {
        "attributes": {},
        "dividend_tax_rate": 0.0,
        "financing": {
          "day_count": "Act360",
          "discount_curve_id": "USD-OIS",
          "forward_curve_id": "USD-SOFR-3M",
          "spread_bp": "75"
        },
        "id": "TRS-SPX-1Y",
        "notional": {
          "amount": "5000000",
          "currency": "USD"
        },
        "pricing_overrides": {},
        "schedule": {
          "end": "2026-01-15",
          "params": {
            "bdc": "following",
            "calendar_id": "weekends_only",
            "dc": "Act360",
            "end_of_month": false,
            "freq": {
              "count": 3,
              "unit": "months"
            },
            "payment_lag_days": 0,
            "stub": "None"
          },
          "start": "2025-01-15"
        },
        "side": "receive_total_return",
        "underlying": {
          "contract_size": 1.0,
          "currency": "USD",
          "div_yield_id": "SPX-DIV",
          "spot_id": "SPX-SPOT",
          "ticker": "SPX"
        }
      },
      "type": "trs_equity"
    },
    "LeveredRealEstateEquity": {
      "spec": {
        "asset": {
          "attributes": {},
          "currency": "USD",
          "day_count": "Act365F",
          "discount_curve_id": "USD-OIS",
          "discount_rate": 0.08,
          "id": "RE-OFFICE-DCF",
          "noi_schedule": [
            [
              "2026-01-01",
              100000.0
            ],
            [
              "2027-01-01",
              100000.0
            ],
            [
              "2028-01-01",
              100000.0
            ],
            [
              "2029-01-01",
              100000.0
            ],
            [
              "2030-01-01",
              100000.0
            ]
          ],
          "pricing_overrides": {
            "adaptive_bumps": false,
            "call_friction_cents": null,
            "cds_quote_bp": null,
            "credit_spread_bump_bp": null,
            "implied_volatility": null,
            "mc_seed_scenario": null,
            "mean_reversion": null,
            "quoted_clean_price": null,
            "rate_bump_bp": null,
            "rho_bump_decimal": null,
            "spot_bump_pct": null,
            "theta_period": null,
            "tree_steps": null,
            "upfront_payment": null,
            "use_gobet_miri": false,
            "vega_bump_decimal": null,
            "vol_bump_pct": null,
            "vol_surface_extrapolation": "error",
            "ytm_bump_decimal": null
          },
          "property_type": "office",
          "terminal_cap_rate": 0.055,
          "valuation_date": "2025-01-01",
          "valuation_method": "dcf"
        },
        "attributes": {},
        "currency": "USD",
        "discount_curve_id": "USD-OIS",
        "id": "RE-LEVERED-OFFICE",
        "pricing_overrides": {
          "adaptive_bumps": false,
          "call_friction_cents": null,
          "cds_quote_bp": null,
          "credit_spread_bump_bp": null,
          "implied_volatility": null,
          "mc_seed_scenario": null,
          "mean_reversion": null,
          "quoted_clean_price": null,
          "rate_bump_bp": null,
          "rho_bump_decimal": null,
          "spot_bump_pct": null,
          "theta_period": null,
          "tree_steps": null,
          "upfront_payment": null,
          "use_gobet_miri": false,
          "vega_bump_decimal": null,
          "vol_bump_pct": null,
          "vol_surface_extrapolation": "error",
          "ytm_bump_decimal": null
        }
      },
      "type": "levered_real_estate_equity"
    },
    "PrivateMarketsFund": {
      "spec": {
        "attributes": {
          "meta": {},
          "tags": []
        },
        "currency": "USD",
        "discount_curve_id": "USD-OIS",
        "events": [
          {
            "amount": {
              "amount": "5000000",
              "currency": "USD"
            },
            "date": "2024-01-15",
            "deal_id": null,
            "kind": "contribution"
          },
          {
            "amount": {
              "amount": "2000000",
              "currency": "USD"
            },
            "date": "2024-06-15",
            "deal_id": null,
            "kind": "contribution"
          },
          {
            "amount": {
              "amount": "4000000",
              "currency": "USD"
            },
            "date": "2026-03-01",
            "deal_id": "DEAL-1",
            "kind": "proceeds"
          },
          {
            "amount": {
              "amount": "4000000",
              "currency": "USD"
            },
            "date": "2027-01-01",
            "deal_id": null,
            "kind": "distribution"
          }
        ],
        "id": "PMF-EXAMPLE",
        "waterfall_spec": {
          "catchup_mode": "full",
          "clawback": null,
          "irr_basis": "Act365F",
          "style": "european",
          "tranches": [
            "return_of_capital",
            {
              "preferred_irr": {
                "irr": 0.08
              }
            },
            {
              "catch_up": {
                "gp_share": 0.5
              }
            },
            {
              "promote_tier": {
                "gp_share": 0.2,
                "hurdle": {
                  "irr": {
                    "rate": 0.12
                  }
                },
                "lp_share": 0.8
              }
            }
          ]
        }
      },
      "type": "private_markets_fund"
    },
    "RealEstateAsset": {
      "spec": {
        "attributes": {},
        "currency": "USD",
        "day_count": "Act365F",
        "discount_curve_id": "USD-OIS",
        "discount_rate": 0.08,
        "id": "RE-OFFICE-DCF",
        "noi_schedule": [
          [
            "2026-01-01",
            100000.0
          ],
          [
            "2027-01-01",
            100000.0
          ],
          [
            "2028-01-01",
            100000.0
          ],
          [
            "2029-01-01",
            100000.0
          ],
          [
            "2030-01-01",
            100000.0
          ]
        ],
        "pricing_overrides": {
          "adaptive_bumps": false,
          "call_friction_cents": null,
          "cds_quote_bp": null,
          "credit_spread_bump_bp": null,
          "implied_volatility": null,
          "mc_seed_scenario": null,
          "mean_reversion": null,
          "quoted_clean_price": null,
          "rate_bump_bp": null,
          "rho_bump_decimal": null,
          "spot_bump_pct": null,
          "theta_period": null,
          "tree_steps": null,
          "upfront_payment": null,
          "use_gobet_miri": false,
          "vega_bump_decimal": null,
          "vol_bump_pct": null,
          "vol_surface_extrapolation": "error",
          "ytm_bump_decimal": null
        },
        "property_type": "office",
        "terminal_cap_rate": 0.055,
        "valuation_date": "2025-01-01",
        "valuation_method": "dcf"
      },
      "type": "real_estate_asset"
    },
    "VarianceSwap": {
      "spec": {
        "attributes": {},
        "day_count": "Act365F",
        "discount_curve_id": "USD-OIS",
        "id": "VARSPX-1Y",
        "maturity": "2026-01-15",
        "notional": {
          "amount": "1000000",
          "currency": "USD"
        },
        "observation_freq": {
          "count": 1,
          "unit": "days"
        },
        "pricing_overrides": {},
        "realized_var_method": "CloseToClose",
        "side": "receive",
        "start_date": "2025-01-15",
        "strike_variance": 0.04,
        "underlying_ticker": "SPX"
      },
      "type": "variance_swap"
    },
    "VolatilityIndexFuture": {
      "spec": {
        "attributes": {},
        "contract_specs": {
          "index_id": "VIX",
          "multiplier": 1000.0,
          "tick_size": 0.05,
          "tick_value": 50.0
        },
        "discount_curve_id": "USD-OIS",
        "expiry": "2025-03-19",
        "id": "VIX-FUT-2025M03",
        "notional": {
          "amount": "100000",
          "currency": "USD"
        },
        "position": "long",
        "pricing_overrides": {
          "adaptive_bumps": false,
          "call_friction_cents": null,
          "cds_quote_bp": null,
          "credit_spread_bump_bp": null,
          "implied_volatility": null,
          "mc_seed_scenario": null,
          "mean_reversion": null,
          "quoted_clean_price": null,
          "rate_bump_bp": null,
          "rho_bump_decimal": null,
          "spot_bump_pct": null,
          "theta_period": null,
          "tree_steps": null,
          "upfront_payment": null,
          "use_gobet_miri": false,
          "vega_bump_decimal": null,
          "vol_bump_pct": null,
          "vol_surface_extrapolation": "error",
          "ytm_bump_decimal": null
        },
        "quoted_price": 21.5,
        "settlement_date": "2025-03-19",
        "vol_index_curve_id": "VIX"
      },
      "type": "volatility_index_future"
    },
    "VolatilityIndexOption": {
      "spec": {
        "attributes": {},
        "contract_specs": {
          "index_id": "VIX",
          "multiplier": 100.0
        },
        "day_count": "Act365F",
        "discount_curve_id": "USD-OIS",
        "exercise_style": "european",
        "expiry": "2025-03-19",
        "id": "VIX-CALL-20-2025M03",
        "notional": {
          "amount": "10000",
          "currency": "USD"
        },
        "option_type": "call",
        "pricing_overrides": {
          "adaptive_bumps": false,
          "call_friction_cents": null,
          "cds_quote_bp": null,
          "credit_spread_bump_bp": null,
          "implied_volatility": null,
          "mc_seed_scenario": null,
          "mean_reversion": null,
          "quoted_clean_price": null,
          "rate_bump_bp": null,
          "rho_bump_decimal": null,
          "spot_bump_pct": null,
          "theta_period": null,
          "tree_steps": null,
          "upfront_payment": null,
          "use_gobet_miri": false,
          "vega_bump_decimal": null,
          "vol_bump_pct": null,
          "vol_surface_extrapolation": "error",
          "ytm_bump_decimal": null
        },
        "strike": 20.0,
        "vol_index_curve_id": "VIX",
        "vol_of_vol_surface_id": "VIX-VOLVOL"
      },
      "type": "volatility_index_option"
    }
  },
  "exotics": {
    "AsianOption": {
      "spec": {
        "attributes": {
          "meta": {},
          "tags": []
        },
        "averaging_method": "arithmetic",
        "day_count": "Act365F",
        "discount_curve_id": "USD-OIS",
        "div_yield_id": "SPX-DIV",
        "expiry": "2024-06-30",
        "fixing_dates": [
          "2024-01-31",
          "2024-02-29",
          "2024-03-31",
          "2024-04-30",
          "2024-05-31",
          "2024-06-30"
        ],
        "id": "ASIAN-SPX-ARITH-6M",
        "notional": {
          "amount": "100000",
          "currency": "USD"
        },
        "option_type": "call",
        "past_fixings": [],
        "pricing_overrides": {
          "adaptive_bumps": false,
          "cds_quote_bp": null,
          "implied_volatility": null,
          "mc_seed_scenario": null,
          "quoted_clean_price": null,
          "rate_bump_bp": null,
          "rho_bump_decimal": null,
          "spot_bump_pct": null,
          "theta_period": null,
          "upfront_payment": null,
          "vega_bump_decimal": null,
          "vol_bump_pct": null,
          "ytm_bump_decimal": null
        },
        "spot_id": "SPX-SPOT",
        "strike": 4500.0,
        "underlying_ticker": "SPX",
        "vol_surface_id": "SPX-VOL"
      },
      "type": "asian_option"
    },
    "BarrierOption": {
      "spec": {
        "attributes": {
          "meta": {},
          "tags": [
            "golden"
          ]
        },
        "barrier": {
          "amount": "1",
          "currency": "USD"
        },
        "barrier_type": "down_and_out",
        "day_count": "Act365F",
        "discount_curve_id": "USD-OIS",
        "div_yield_id": "SPX-DIVYIELD",
        "expiry": "2027-04-30",
        "id": "BAR-SPX-DOW-DEGEN-GOLDEN",
        "notional": {
          "amount": "1",
          "currency": "USD"
        },
        "option_type": "call",
        "spot_id": "SPX-SPOT",
        "strike": 100.0,
        "underlying_ticker": "SPX",
        "use_gobet_miri": false,
        "vol_surface_id": "SPX-VOL"
      },
      "type": "barrier_option"
    },
    "Basket": {
      "spec": {
        "attributes": {
          "meta": {},
          "tags": []
        },
        "constituents": [
          {
            "id": "EQ-AAPL",
            "reference": {
              "asset_type": "equity",
              "price_id": "AAPL-SPOT"
            },
            "ticker": "AAPL",
            "units": null,
            "weight": 0.6
          },
          {
            "id": "BOND-UST10",
            "reference": {
              "asset_type": "bond",
              "price_id": "UST10Y-PRICE"
            },
            "ticker": "UST10Y",
            "units": null,
            "weight": 0.4
          }
        ],
        "currency": "USD",
        "discount_curve_id": "USD-OIS",
        "expense_ratio": 0.0025,
        "id": "BASKET-60-40",
        "notional": {
          "amount": "1000000",
          "currency": "USD"
        },
        "pricing_config": {
          "days_in_year": 365.25,
          "fx_policy": "cashflow_date"
        }
      },
      "type": "basket"
    },
    "LookbackOption": {
      "spec": {
        "attributes": {
          "meta": {},
          "tags": []
        },
        "day_count": "Act365F",
        "discount_curve_id": "USD-OIS",
        "div_yield_id": "SPX-DIV",
        "expiry": "2024-12-20",
        "id": "LOOKBACK-SPX-FIXED-CALL",
        "lookback_type": "fixed_strike",
        "notional": {
          "amount": "100000",
          "currency": "USD"
        },
        "option_type": "call",
        "pricing_overrides": {
          "adaptive_bumps": false,
          "cds_quote_bp": null,
          "implied_volatility": null,
          "mc_seed_scenario": null,
          "quoted_clean_price": null,
          "rate_bump_bp": null,
          "rho_bump_decimal": null,
          "spot_bump_pct": null,
          "theta_period": null,
          "upfront_payment": null,
          "vega_bump_decimal": null,
          "vol_bump_pct": null,
          "ytm_bump_decimal": null
        },
        "spot_id": "SPX-SPOT",
        "strike": 4500.0,
        "underlying_ticker": "SPX",
        "vol_surface_id": "SPX-VOL"
      },
      "type": "lookback_option"
    }
  },
  "fixed_income": {
    "AgencyCmo": {
      "spec": {
        "agency": "FNMA",
        "attributes": {
          "meta": {
            "deal": "fnr-2024-1"
          },
          "tags": [
            "agency",
            "cmo"
          ]
        },
        "collateral_wac": 0.045,
        "collateral_wam": 360,
        "deal_name": "FNR 2024-1",
        "discount_curve_id": "USD-OIS",
        "id": "FNR-2024-1-A",
        "issue_date": "2024-01-01",
        "pricing_overrides": {
          "adaptive_bumps": false,
          "call_friction_cents": null,
          "cds_quote_bp": null,
          "credit_spread_bump_bp": null,
          "implied_volatility": null,
          "mc_seed_scenario": null,
          "mean_reversion": null,
          "quoted_clean_price": null,
          "rate_bump_bp": null,
          "rho_bump_decimal": null,
          "spot_bump_pct": null,
          "theta_period": null,
          "tree_steps": null,
          "upfront_payment": null,
          "use_gobet_miri": false,
          "vega_bump_decimal": null,
          "vol_bump_pct": null,
          "vol_surface_extrapolation": "error",
          "ytm_bump_decimal": null
        },
        "reference_tranche_id": "A",
        "waterfall": {
          "pro_rata_same_priority": false,
          "tranches": [
            {
              "coupon": 0.04,
              "current_face": {
                "amount": "40000000",
                "currency": "USD"
              },
              "id": "A",
              "original_face": {
                "amount": "40000000",
                "currency": "USD"
              },
              "priority": 1,
              "tranche_type": "sequential"
            },
            {
              "coupon": 0.045,
              "current_face": {
                "amount": "30000000",
                "currency": "USD"
              },
              "id": "B",
              "original_face": {
                "amount": "30000000",
                "currency": "USD"
              },
              "priority": 2,
              "tranche_type": "sequential"
            },
            {
              "coupon": 0.05,
              "current_face": {
                "amount": "30000000",
                "currency": "USD"
              },
              "id": "Z",
              "original_face": {
                "amount": "30000000",
                "currency": "USD"
              },
              "priority": 3,
              "tranche_type": "sequential"
            }
          ]
        }
      },
      "type": "agency_cmo"
    },
    "AgencyMbsPassthrough": {
      "spec": {
        "agency": "FNMA",
        "attributes": {
          "meta": {
            "program": "fnma"
          },
          "tags": [
            "agency",
            "mbs"
          ]
        },
        "current_face": {
          "amount": "950000",
          "currency": "USD"
        },
        "current_factor": 0.95,
        "day_count": "Thirty360",
        "discount_curve_id": "USD-OIS",
        "guarantee_fee_rate": 0.0025,
        "id": "FN-MA1234",
        "issue_date": "2022-01-01",
        "maturity": "2052-01-01",
        "original_face": {
          "amount": "1000000",
          "currency": "USD"
        },
        "pass_through_rate": 0.04,
        "pool_id": "MA1234",
        "pool_type": "generic",
        "prepayment_model": {
          "cpr": 0.06,
          "curve": {
            "curve": "psa",
            "speed_multiplier": 1.0
          }
        },
        "pricing_overrides": {
          "adaptive_bumps": false,
          "call_friction_cents": null,
          "cds_quote_bp": null,
          "credit_spread_bump_bp": null,
          "implied_volatility": null,
          "mc_seed_scenario": null,
          "mean_reversion": null,
          "quoted_clean_price": null,
          "rate_bump_bp": null,
          "rho_bump_decimal": null,
          "spot_bump_pct": null,
          "theta_period": null,
          "tree_steps": null,
          "upfront_payment": null,
          "use_gobet_miri": false,
          "vega_bump_decimal": null,
          "vol_bump_pct": null,
          "vol_surface_extrapolation": "error",
          "ytm_bump_decimal": null
        },
        "servicing_fee_rate": 0.0025,
        "wac": 0.045,
        "wam": 348
      },
      "type": "agency_mbs_passthrough"
    },
    "AgencyTba": {
      "spec": {
        "agency": "FNMA",
        "attributes": {
          "meta": {
            "program": "fnma"
          },
          "tags": [
            "agency",
            "tba"
          ]
        },
        "coupon": 0.04,
        "discount_curve_id": "USD-OIS",
        "id": "FN30-4.0-202703",
        "notional": {
          "amount": "10000000",
          "currency": "USD"
        },
        "pricing_overrides": {
          "adaptive_bumps": false,
          "call_friction_cents": null,
          "cds_quote_bp": null,
          "credit_spread_bump_bp": null,
          "implied_volatility": null,
          "mc_seed_scenario": null,
          "mean_reversion": null,
          "quoted_clean_price": null,
          "rate_bump_bp": null,
          "rho_bump_decimal": null,
          "spot_bump_pct": null,
          "theta_period": null,
          "tree_steps": null,
          "upfront_payment": null,
          "use_gobet_miri": false,
          "vega_bump_decimal": null,
          "vol_bump_pct": null,
          "vol_surface_extrapolation": "error",
          "ytm_bump_decimal": null
        },
        "settlement_month": 3,
        "settlement_year": 2027,
        "term": "thirty_year",
        "trade_price": 98.5
      },
      "type": "agency_tba"
    },
    "Bond": {
      "spec": {
        "accrual_method": "Linear",
        "attributes": {
          "meta": {},
          "tags": []
        },
        "call_put": null,
        "cashflow_spec": {
          "Fixed": {
            "bdc": "following",
            "calendar_id": "weekends_only",
            "coupon_type": "Cash",
            "dc": "Thirty360",
            "end_of_month": false,
            "freq": {
              "count": 6,
              "unit": "months"
            },
            "payment_lag_days": 0,
            "rate": "0.0425",
            "stub": "None"
          }
        },
        "credit_curve_id": null,
        "discount_curve_id": "USD-OIS",
        "ex_coupon_days": 0,
        "id": "UST-FIXED-10Y",
        "issue_date": "2024-01-15",
        "maturity": "2034-01-15",
        "notional": {
          "amount": "1000000",
          "currency": "USD"
        },
        "pricing_overrides": {
          "quoted_clean_price": 98.75
        },
        "settlement_days": 1
      },
      "type": "bond"
    },
    "BondFuture": {
      "spec": {
        "attributes": {
          "meta": {},
          "tags": [
            "golden"
          ]
        },
        "contract_specs": {
          "calendar_id": "nyse",
          "contract_size": 100000.0,
          "repo_day_count_basis": "act360",
          "settlement_days": 2,
          "standard_coupon": 0.06,
          "standard_maturity_years": 10.0,
          "tick_size": 0.015625,
          "tick_value": 15.625
        },
        "ctd_bond": {
          "accrual_method": "Linear",
          "attributes": {
            "meta": {},
            "tags": [
              "golden"
            ]
          },
          "call_put": null,
          "cashflow_spec": {
            "Fixed": {
              "calendar_id": "nyse",
              "dc": "Thirty360",
              "freq": {
                "count": 6,
                "unit": "months"
              },
              "rate": "0.0425",
              "stub": "None"
            }
          },
          "credit_curve_id": null,
          "discount_curve_id": "USD-TREASURY",
          "ex_coupon_days": 0,
          "id": "US91282CJL54",
          "issue_date": "2024-05-15",
          "maturity": "2034-05-15",
          "notional": {
            "amount": "100000",
            "currency": "USD"
          },
          "pricing_overrides": {
            "quoted_clean_price": 98.5
          },
          "settlement_days": 1
        },
        "ctd_bond_id": "US91282CJL54",
        "deliverable_basket": [
          {
            "bond_id": "US91282CJL54",
            "conversion_factor": 0.8234
          }
        ],
        "delivery_end": "2026-06-30",
        "delivery_start": "2026-06-22",
        "discount_curve_id": "USD-TREASURY",
        "expiry": "2026-06-19",
        "id": "UST-TY-10Y-FRONT-MONTH-GOLDEN",
        "notional": {
          "amount": "1000000",
          "currency": "USD"
        },
        "position": "long",
        "quoted_price": 119.5
      },
      "type": "bond_future"
    },
    "ConvertibleBond": {
      "spec": {
        "attributes": {
          "meta": {},
          "tags": []
        },
        "conversion": {
          "anti_dilution": "None",
          "dividend_adjustment": "None",
          "policy": "Voluntary",
          "ratio": 25.0
        },
        "credit_curve_id": "USD-CREDIT-BBB",
        "discount_curve_id": "USD-IG",
        "fixed_coupon": {
          "bdc": "following",
          "calendar_id": "weekends_only",
          "coupon_type": "Cash",
          "dc": "Thirty360",
          "end_of_month": false,
          "freq": {
            "count": 6,
            "unit": "months"
          },
          "payment_lag_days": 0,
          "rate": 0.02,
          "stub": "None"
        },
        "id": "CB-TECH-5Y",
        "issue_date": "2024-01-15",
        "maturity": "2029-01-15",
        "notional": {
          "amount": "1000000",
          "currency": "USD"
        },
        "pricing_overrides": {},
        "underlying_equity_id": "TECH"
      },
      "type": "convertible_bond"
    },
    "DollarRoll": {
      "spec": {
        "agency": "FNMA",
        "attributes": {
          "meta": {
            "program": "fnma"
          },
          "tags": [
            "agency",
            "dollar_roll"
          ]
        },
        "back_price": 98.0,
        "back_settlement_month": 4,
        "back_settlement_year": 2024,
        "coupon": 0.04,
        "discount_curve_id": "USD-OIS",
        "front_price": 98.5,
        "front_settlement_month": 3,
        "front_settlement_year": 2024,
        "id": "FN30-4.0-ROLL-0324-0424",
        "notional": {
          "amount": "10000000",
          "currency": "USD"
        },
        "pricing_overrides": {
          "adaptive_bumps": false,
          "call_friction_cents": null,
          "cds_quote_bp": null,
          "credit_spread_bump_bp": null,
          "implied_volatility": null,
          "mc_seed_scenario": null,
          "mean_reversion": null,
          "quoted_clean_price": null,
          "rate_bump_bp": null,
          "rho_bump_decimal": null,
          "spot_bump_pct": null,
          "theta_period": null,
          "tree_steps": null,
          "upfront_payment": null,
          "use_gobet_miri": false,
          "vega_bump_decimal": null,
          "vol_bump_pct": null,
          "vol_surface_extrapolation": "error",
          "ytm_bump_decimal": null
        },
        "term": "thirty_year"
      },
      "type": "dollar_roll"
    },
    "FIIndexTotalReturnSwap": {
      "spec": {
        "attributes": {},
        "financing": {
          "day_count": "Act360",
          "discount_curve_id": "USD-OIS",
          "forward_curve_id": "USD-SOFR-3M",
          "spread_bp": "100"
        },
        "id": "TRS-US-CORP-1Y",
        "initial_level": null,
        "notional": {
          "amount": "5000000",
          "currency": "USD"
        },
        "pricing_overrides": {
          "adaptive_bumps": false,
          "call_friction_cents": null,
          "cds_quote_bp": null,
          "credit_spread_bump_bp": null,
          "implied_volatility": null,
          "mc_seed_scenario": null,
          "mean_reversion": null,
          "quoted_clean_price": null,
          "rate_bump_bp": null,
          "rho_bump_decimal": null,
          "spot_bump_pct": null,
          "theta_period": null,
          "tree_steps": null,
          "upfront_payment": null,
          "use_gobet_miri": false,
          "vega_bump_decimal": null,
          "vol_bump_pct": null,
          "vol_surface_extrapolation": "error",
          "ytm_bump_decimal": null
        },
        "schedule": {
          "end": "2025-04-15",
          "params": {
            "bdc": "following",
            "calendar_id": "weekends_only",
            "dc": "Act360",
            "end_of_month": false,
            "freq": {
              "count": 3,
              "unit": "months"
            },
            "payment_lag_days": 0,
            "stub": "None"
          },
          "start": "2024-04-15"
        },
        "side": "receive_total_return",
        "underlying": {
          "base_currency": "USD",
          "contract_size": 1.0,
          "convexity_id": "US-CORP-CONVEXITY",
          "duration_id": "US-CORP-DURATION",
          "index_id": "US-CORP-INDEX",
          "yield_id": "US-CORP-YIELD"
        }
      },
      "type": "trs_fixed_income_index"
    },
    "InflationLinkedBond": {
      "spec": {
        "attributes": {
          "meta": {},
          "tags": []
        },
        "base_date": "2024-01-15",
        "base_index": 277.0676975464149,
        "bdc": "following",
        "calendar_id": "weekends_only",
        "day_count": "Thirty360",
        "deflation_protection": "MaturityOnly",
        "discount_curve_id": "USD-TIPS",
        "frequency": {
          "count": 6,
          "unit": "months"
        },
        "id": "TIPS-DEMO",
        "indexation_method": "TIPS",
        "inflation_index_id": "US-CPI",
        "issue_date": "2024-01-15",
        "lag": {
          "months": 3
        },
        "maturity": "2034-01-15",
        "notional": {
          "amount": "1000000",
          "currency": "USD"
        },
        "pricing_overrides": {},
        "quoted_clean": 100.0,
        "real_coupon": "0.025",
        "stub": "None"
      },
      "type": "inflation_linked_bond"
    },
    "RevolvingCredit": {
      "spec": {
        "attributes": {
          "meta": {},
          "tags": []
        },
        "base_rate_spec": {
          "Floating": {
            "all_in_floor_bp": null,
            "bdc": "modified_following",
            "calendar_id": "weekends_only",
            "cap_bp": null,
            "dc": "Act360",
            "end_of_month": false,
            "fixing_calendar_id": null,
            "floor_bp": "0",
            "gearing": "1",
            "gearing_includes_spread": true,
            "index_cap_bp": null,
            "index_id": "USD-SOFR-3M",
            "payment_lag_days": 0,
            "reset_freq": {
              "count": 3,
              "unit": "months"
            },
            "reset_lag_days": 2,
            "spread_bp": "250"
          }
        },
        "commitment_amount": {
          "amount": "50000000",
          "currency": "USD"
        },
        "commitment_date": "2024-01-01",
        "day_count": "Act360",
        "discount_curve_id": "USD-OIS",
        "draw_repay_spec": {
          "Deterministic": [
            {
              "amount": {
                "amount": "5000000",
                "currency": "USD"
              },
              "date": "2024-03-01",
              "is_draw": true
            },
            {
              "amount": {
                "amount": "3000000",
                "currency": "USD"
              },
              "date": "2025-06-01",
              "is_draw": false
            }
          ]
        },
        "drawn_amount": {
          "amount": "10000000",
          "currency": "USD"
        },
        "fees": {
          "commitment_fee_tiers": [
            {
              "bps": "25",
              "threshold": "0"
            }
          ],
          "facility_fee_bp": 5.0,
          "usage_fee_tiers": [
            {
              "bps": "10",
              "threshold": "0"
            }
          ]
        },
        "frequency": {
          "count": 3,
          "unit": "months"
        },
        "id": "RCF-USD-3Y",
        "maturity": "2027-01-01",
        "pricing_overrides": {},
        "recovery_rate": 0.7,
        "stub": "ShortFront"
      },
      "type": "revolving_credit"
    },
    "StructuredCredit": {
      "spec": {
        "attributes": {
          "meta": {
            "book": "structured_credit"
          },
          "tags": [
            "full"
          ]
        },
        "behavior_overrides": {
          "abs_speed": 0.012,
          "cdr_annual": 0.025,
          "cpr_annual": 0.18,
          "psa_speed_multiplier": 1.3,
          "recovery_lag_months": 9,
          "recovery_rate": 0.42,
          "reinvestment_price": 101.0,
          "sda_speed_multiplier": 1.1
        },
        "closing_date": "2024-01-01",
        "correlation_structure": {
          "inter_sector": 0.12,
          "intra_sector": 0.28,
          "prepay_default": -0.18,
          "structure": "Sectored"
        },
        "credit_factors": {
          "credit_score": 720,
          "custom_factors": {
            "fico_band": 700.0
          },
          "delinquency_days": 15,
          "dti": 0.32,
          "ltv": 0.85,
          "unemployment_rate": 0.04
        },
        "deal_metadata": {
          "manager_id": "Manager-X",
          "master_servicer_id": "Master-Z",
          "servicer_id": "Servicer-Y",
          "special_servicer_id": "Special-W",
          "trustee_id": "Trustee-T"
        },
        "deal_type": "CLO",
        "default_assumptions": {
          "abs_speed_monthly": 0.011,
          "base_cdr_annual": 0.03,
          "base_cpr_annual": 0.09,
          "base_recovery_rate": 0.45,
          "cdr_by_asset_type": {
            "abs_auto": 0.03
          },
          "cpr_by_asset_type": {
            "abs_auto": 0.2
          },
          "psa_speed": 110.0,
          "recovery_by_asset_type": {
            "abs_auto": 0.5
          },
          "sda_speed": 1.2
        },
        "default_spec": {
          "cdr": 0.03,
          "curve": {
            "curve": "sda",
            "speed_multiplier": 125.0
          }
        },
        "discount_curve_id": "USD-SOFR-DISC",
        "first_payment_date": "2024-04-01",
        "frequency": {
          "count": 1,
          "unit": "months"
        },
        "hedge_swaps": [
          {
            "attributes": {
              "meta": {},
              "tags": []
            },
            "fixed": {
              "bdc": "modified_following",
              "calendar_id": "USD",
              "compounding_simple": true,
              "day_count": "Thirty360",
              "discount_curve_id": "USD-OIS",
              "end": "2028-01-01",
              "frequency": {
                "count": 6,
                "unit": "months"
              },
              "par_method": null,
              "rate": 0.015,
              "start": "2024-01-01",
              "stub": "None"
            },
            "float": {
              "bdc": "modified_following",
              "calendar_id": "USD",
              "compounding": "Simple",
              "day_count": "Act360",
              "discount_curve_id": "USD-OIS",
              "end": "2028-01-01",
              "fixing_calendar_id": null,
              "forward_curve_id": "USD-SOFR-3M",
              "frequency": {
                "count": 3,
                "unit": "months"
              },
              "reset_lag_days": 2,
              "spread_bp": 0.0,
              "start": "2024-01-01",
              "stub": "None"
            },
            "id": "HEDGE-SWAP",
            "notional": {
              "amount": "10000000",
              "currency": "USD"
            },
            "side": "pay"
          }
        ],
        "id": "FULL-CLO",
        "market_conditions": {
          "custom_factors": {
            "stress": 1.2
          },
          "hpa": 0.02,
          "original_rate": 0.05,
          "refi_rate": 0.035,
          "seasonal_factor": 0.98,
          "unemployment": 0.05
        },
        "maturity": "2034-01-01",
        "payment_calendar_id": "nyse",
        "pool": {
          "assets": [
            {
              "acquisition_date": null,
              "asset_type": {
                "industry": null,
                "type": "FirstLienLoan"
              },
              "balance": {
                "amount": "12000000",
                "currency": "USD"
              },
              "credit_quality": "BB",
              "day_count": "Act360",
              "id": "LOAN1",
              "index_id": "SOFR-3M",
              "industry": "Technology",
              "is_defaulted": false,
              "maturity": "2030-01-01",
              "mdr_override": 0.0042,
              "obligor_id": "OBLIGOR-1",
              "purchase_price": null,
              "rate": 0.035,
              "recovery_amount": null,
              "smm_override": 0.0123,
              "spread_bps": 350.0
            },
            {
              "acquisition_date": null,
              "asset_type": {
                "industry": null,
                "type": "HighYieldBond"
              },
              "balance": {
                "amount": "8000000",
                "currency": "USD"
              },
              "credit_quality": "A",
              "day_count": "Act365F",
              "id": "BOND1",
              "index_id": null,
              "industry": "Healthcare",
              "is_defaulted": true,
              "maturity": "2029-07-01",
              "mdr_override": null,
              "obligor_id": "OBLIGOR-2",
              "purchase_price": {
                "amount": "7800000",
                "currency": "USD"
              },
              "rate": 0.055,
              "recovery_amount": {
                "amount": "1000000",
                "currency": "USD"
              },
              "smm_override": null,
              "spread_bps": null
            }
          ],
          "base_currency": "USD",
          "collection_account": {
            "amount": "250000",
            "currency": "USD"
          },
          "cumulative_defaults": {
            "amount": "0",
            "currency": "USD"
          },
          "cumulative_prepayments": {
            "amount": "0",
            "currency": "USD"
          },
          "cumulative_recoveries": {
            "amount": "0",
            "currency": "USD"
          },
          "deal_type": "CLO",
          "excess_spread_account": {
            "amount": "75000",
            "currency": "USD"
          },
          "id": "POOL-FULL",
          "reinvestment_period": {
            "criteria": {
              "apply_eligibility_criteria": true,
              "maintain_credit_quality": true,
              "maintain_wal": false,
              "max_price": 102.5,
              "min_yield": 0.04
            },
            "end_date": "2026-01-01",
            "is_active": true
          },
          "rep_lines": [
            {
              "balance": {
                "amount": "20000000",
                "currency": "USD"
              },
              "cdr": 0.03,
              "cpr": 0.08,
              "day_count": "Act360",
              "id": "REP1",
              "index_id": "SOFR-1M",
              "maturity": "2031-01-01",
              "rate": 0.055,
              "recovery_rate": 0.5,
              "seasoning_months": 12,
              "spread_bps": 180.0
            }
          ],
          "reserve_account": {
            "amount": "100000",
            "currency": "USD"
          }
        },
        "prepayment_spec": {
          "cpr": 0.06,
          "curve": {
            "curve": "psa",
            "speed_multiplier": 175.0
          }
        },
        "recovery_spec": {
          "rate": 0.55,
          "recovery_lag": 10
        },
        "reinvestment_end_date": "2026-01-01",
        "stochastic_default_spec": {
          "base_cdr": 0.025,
          "copula_spec": {
            "type": "Gaussian"
          },
          "correlation": 0.35,
          "model": "Copula"
        },
        "stochastic_prepay_spec": {
          "base_spec": {
            "cpr": 0.06,
            "curve": {
              "curve": "psa",
              "speed_multiplier": 1.1
            }
          },
          "cpr_volatility": 0.12,
          "factor_loading": 0.25,
          "model": "FactorCorrelated"
        },
        "tranches": {
          "total_size": {
            "amount": "50000000",
            "currency": "USD"
          },
          "tranches": [
            {
              "attachment_point": 0.0,
              "attributes": {
                "meta": {
                  "desk": "alts"
                },
                "tags": [
                  "equity"
                ]
              },
              "behavior_type": "Standard",
              "can_reinvest": true,
              "coupon": {
                "Fixed": {
                  "rate": 0.12
                }
              },
              "credit_enhancement": {
                "cash_trap_active": false,
                "excess_spread": 0.0,
                "overcollateralization": {
                  "amount": "0",
                  "currency": "USD"
                },
                "reserve_account": {
                  "amount": "0",
                  "currency": "USD"
                },
                "subordination": {
                  "amount": "0",
                  "currency": "USD"
                }
              },
              "current_balance": {
                "amount": "5000000",
                "currency": "USD"
              },
              "day_count": "Act360",
              "deferred_interest": {
                "amount": "0",
                "currency": "USD"
              },
              "detachment_point": 10.0,
              "expected_maturity": "2030-01-01",
              "frequency": {
                "count": 3,
                "unit": "months"
              },
              "ic_trigger": {
                "breach_date": null,
                "consequence": "DivertCashFlow",
                "cure_level": null,
                "trigger_level": 1.05
              },
              "id": "EQUITY",
              "is_revolving": true,
              "maturity": "2034-01-01",
              "oc_trigger": {
                "breach_date": null,
                "consequence": "DivertCashFlow",
                "cure_level": null,
                "trigger_level": 1.15
              },
              "original_balance": {
                "amount": "5000000",
                "currency": "USD"
              },
              "payment_priority": 2,
              "rating": "BB",
              "seniority": "Equity",
              "target_balance": null
            },
            {
              "attachment_point": 10.0,
              "attributes": {
                "meta": {},
                "tags": [
                  "senior"
                ]
              },
              "behavior_type": "Standard",
              "can_reinvest": false,
              "coupon": {
                "Floating": {
                  "all_in_floor_bp": 25.0,
                  "bdc": "modified_following",
                  "calendar_id": "NYC",
                  "cap_bp": 1200.0,
                  "dc": "Act360",
                  "end_of_month": false,
                  "fixing_calendar_id": "NYC",
                  "floor_bp": 0.0,
                  "gearing": 1.0,
                  "gearing_includes_spread": true,
                  "index_cap_bp": null,
                  "index_id": "SOFR-3M",
                  "payment_lag_days": 0,
                  "reset_freq": {
                    "count": 3,
                    "unit": "months"
                  },
                  "reset_lag_days": 2,
                  "spread_bp": 150.0
                }
              },
              "credit_enhancement": {
                "cash_trap_active": false,
                "excess_spread": 0.0,
                "overcollateralization": {
                  "amount": "0",
                  "currency": "USD"
                },
                "reserve_account": {
                  "amount": "0",
                  "currency": "USD"
                },
                "subordination": {
                  "amount": "0",
                  "currency": "USD"
                }
              },
              "current_balance": {
                "amount": "45000000",
                "currency": "USD"
              },
              "day_count": "Act360",
              "deferred_interest": {
                "amount": "0",
                "currency": "USD"
              },
              "detachment_point": 100.0,
              "expected_maturity": null,
              "frequency": {
                "count": 3,
                "unit": "months"
              },
              "ic_trigger": null,
              "id": "SENIOR",
              "is_revolving": false,
              "maturity": "2034-01-01",
              "oc_trigger": null,
              "original_balance": {
                "amount": "45000000",
                "currency": "USD"
              },
              "payment_priority": 1,
              "rating": "AAA",
              "seniority": "Senior",
              "target_balance": {
                "amount": "30000000",
                "currency": "USD"
              }
            }
          ]
        }
      },
      "type": "structured_credit"
    },
    "TermLoan": {
      "spec": {
        "amortization": {
          "PercentPerPeriod": {
            "bp": 250
          }
        },
        "attributes": {
          "meta": {},
          "tags": []
        },
        "bdc": "modified_following",
        "calendar_id": null,
        "coupon_type": "Cash",
        "currency": "USD",
        "day_count": "Act360",
        "discount_curve_id": "USD-OIS",
        "frequency": {
          "count": 3,
          "unit": "months"
        },
        "id": "TERM-LOAN-USD-5Y",
        "issue_date": "2024-01-01",
        "maturity": "2029-01-01",
        "notional_limit": {
          "amount": "10000000",
          "currency": "USD"
        },
        "pricing_overrides": {},
        "rate": {
          "Fixed": {
            "rate_bp": 600
          }
        },
        "settlement_days": 2,
        "stub": "None"
      },
      "type": "term_loan"
    }
  },
  "fx": {
    "FxBarrierOption": {
      "spec": {
        "attributes": {},
        "barrier": 1.2,
        "barrier_type": "up_and_out",
        "base_currency": "EUR",
        "day_count": "Act365F",
        "domestic_discount_curve_id": "USD-OIS",
        "expiry": "2024-12-20",
        "foreign_discount_curve_id": "EUR-OIS",
        "fx_spot_id": "EURUSD-SPOT",
        "id": "FXBAR-EURUSD-UO-CALL",
        "notional": {
          "amount": "1000000",
          "currency": "EUR"
        },
        "option_type": "call",
        "pricing_overrides": {
          "adaptive_bumps": false,
          "call_friction_cents": null,
          "cds_quote_bp": null,
          "credit_spread_bump_bp": null,
          "implied_volatility": null,
          "mc_seed_scenario": null,
          "mean_reversion": null,
          "quoted_clean_price": null,
          "rate_bump_bp": null,
          "rho_bump_decimal": null,
          "spot_bump_pct": null,
          "theta_period": null,
          "tree_steps": null,
          "upfront_payment": null,
          "use_gobet_miri": false,
          "vega_bump_decimal": null,
          "vol_bump_pct": null,
          "vol_surface_extrapolation": "error",
          "ytm_bump_decimal": null
        },
        "quote_currency": "USD",
        "rebate": null,
        "strike": 1.1,
        "use_gobet_miri": false,
        "vol_surface_id": "EURUSD-VOL"
      },
      "type": "fx_barrier_option"
    },
    "FxDigitalOption": {
      "spec": {
        "attributes": {},
        "base_currency": "EUR",
        "day_count": "Act365F",
        "domestic_discount_curve_id": "USD-OIS",
        "expiry": "2024-06-21",
        "foreign_discount_curve_id": "EUR-OIS",
        "id": "FXDIG-EURUSD-CALL",
        "notional": {
          "amount": "1000000",
          "currency": "EUR"
        },
        "option_type": "call",
        "payout_amount": {
          "amount": "1000000",
          "currency": "USD"
        },
        "payout_type": "cash_or_nothing",
        "pricing_overrides": {
          "adaptive_bumps": false,
          "call_friction_cents": null,
          "cds_quote_bp": null,
          "credit_spread_bump_bp": null,
          "implied_volatility": null,
          "mc_seed_scenario": null,
          "mean_reversion": null,
          "quoted_clean_price": null,
          "rate_bump_bp": null,
          "rho_bump_decimal": null,
          "spot_bump_pct": null,
          "theta_period": null,
          "tree_steps": null,
          "upfront_payment": null,
          "use_gobet_miri": false,
          "vega_bump_decimal": null,
          "vol_bump_pct": null,
          "vol_surface_extrapolation": "error",
          "ytm_bump_decimal": null
        },
        "quote_currency": "USD",
        "strike": 1.12,
        "vol_surface_id": "EURUSD-VOL"
      },
      "type": "fx_digital_option"
    },
    "FxForward": {
      "spec": {
        "attributes": {
          "meta": {
            "pair": "EURUSD"
          },
          "tags": [
            "fx"
          ]
        },
        "base_currency": "EUR",
        "contract_rate": 1.12,
        "domestic_discount_curve_id": "USD-OIS",
        "foreign_discount_curve_id": "EUR-OIS",
        "id": "EURUSD-FWD-6M",
        "maturity": "2025-07-15",
        "notional": {
          "amount": "1000000",
          "currency": "EUR"
        },
        "pricing_overrides": {},
        "quote_currency": "USD"
      },
      "type": "fx_forward"
    },
    "FxOption": {
      "spec": {
        "attributes": {
          "meta": {},
          "tags": []
        },
        "base_currency": "EUR",
        "day_count": "Act365F",
        "domestic_discount_curve_id": "USD-OIS",
        "exercise_style": "european",
        "expiry": "2024-06-21",
        "foreign_discount_curve_id": "EUR-OIS",
        "id": "FXOPT-EURUSD-CALL",
        "notional": {
          "amount": "1000000",
          "currency": "EUR"
        },
        "option_type": "call",
        "pricing_overrides": {
          "adaptive_bumps": false,
          "cds_quote_bp": null,
          "implied_volatility": null,
          "mc_seed_scenario": null,
          "quoted_clean_price": null,
          "rate_bump_bp": null,
          "rho_bump_decimal": null,
          "spot_bump_pct": null,
          "theta_period": null,
          "upfront_payment": null,
          "vega_bump_decimal": null,
          "vol_bump_pct": null,
          "ytm_bump_decimal": null
        },
        "quote_currency": "USD",
        "settlement": "cash",
        "strike": 1.12,
        "vol_surface_id": "EURUSD-VOL"
      },
      "type": "fx_option"
    },
    "FxSpot": {
      "spec": {
        "attributes": {},
        "base_currency": "EUR",
        "bdc": "modified_following",
        "id": "EURUSD-SPOT",
        "notional": {
          "amount": "1000000",
          "currency": "EUR"
        },
        "pricing_overrides": {},
        "quote_currency": "USD",
        "settlement_lag_days": 2,
        "spot_rate": 1.1
      },
      "type": "fx_spot"
    },
    "FxSwap": {
      "spec": {
        "attributes": {
          "meta": {},
          "tags": []
        },
        "base_currency": "EUR",
        "base_notional": {
          "amount": "1000000",
          "currency": "EUR"
        },
        "domestic_discount_curve_id": "USD-OIS",
        "far_date": "2024-07-05",
        "far_rate": 1.12,
        "foreign_discount_curve_id": "EUR-OIS",
        "id": "FXSWAP-EURUSD-6M",
        "near_date": "2024-01-05",
        "near_rate": 1.1,
        "quote_currency": "USD"
      },
      "type": "fx_swap"
    },
    "FxTouchOption": {
      "spec": {
        "attributes": {},
        "barrier_direction": "down",
        "barrier_level": 1.05,
        "base_currency": "EUR",
        "day_count": "Act365F",
        "domestic_discount_curve_id": "USD-OIS",
        "expiry": "2024-06-21",
        "foreign_discount_curve_id": "EUR-OIS",
        "id": "FXTOUCH-EURUSD-OT",
        "payout_amount": {
          "amount": "1000000",
          "currency": "USD"
        },
        "payout_timing": "at_expiry",
        "pricing_overrides": {
          "adaptive_bumps": false,
          "call_friction_cents": null,
          "cds_quote_bp": null,
          "credit_spread_bump_bp": null,
          "implied_volatility": null,
          "mc_seed_scenario": null,
          "mean_reversion": null,
          "quoted_clean_price": null,
          "rate_bump_bp": null,
          "rho_bump_decimal": null,
          "spot_bump_pct": null,
          "theta_period": null,
          "tree_steps": null,
          "upfront_payment": null,
          "use_gobet_miri": false,
          "vega_bump_decimal": null,
          "vol_bump_pct": null,
          "vol_surface_extrapolation": "error",
          "ytm_bump_decimal": null
        },
        "quote_currency": "USD",
        "touch_type": "one_touch",
        "vol_surface_id": "EURUSD-VOL"
      },
      "type": "fx_touch_option"
    },
    "FxVarianceSwap": {
      "spec": {
        "attributes": {},
        "base_currency": "EUR",
        "close_series_id": null,
        "day_count": "Act365F",
        "domestic_discount_curve_id": "USD-OIS",
        "foreign_discount_curve_id": "EUR-OIS",
        "high_series_id": null,
        "id": "FXVAR-EURUSD-1Y",
        "low_series_id": null,
        "maturity": "2025-01-02",
        "notional": {
          "amount": "1000000",
          "currency": "USD"
        },
        "observation_freq": {
          "count": 1,
          "unit": "days"
        },
        "open_series_id": null,
        "pricing_overrides": {
          "adaptive_bumps": false,
          "call_friction_cents": null,
          "cds_quote_bp": null,
          "credit_spread_bump_bp": null,
          "implied_volatility": null,
          "mc_seed_scenario": null,
          "mean_reversion": null,
          "quoted_clean_price": null,
          "rate_bump_bp": null,
          "rho_bump_decimal": null,
          "spot_bump_pct": null,
          "theta_period": null,
          "tree_steps": null,
          "upfront_payment": null,
          "use_gobet_miri": false,
          "vega_bump_decimal": null,
          "vol_bump_pct": null,
          "vol_surface_extrapolation": "error",
          "ytm_bump_decimal": null
        },
        "quote_currency": "USD",
        "realized_var_method": "CloseToClose",
        "side": "receive",
        "spot_id": "EURUSD",
        "start_date": "2024-01-02",
        "strike_variance": 0.04,
        "vol_surface_id": "EURUSD-VOL"
      },
      "type": "fx_variance_swap"
    },
    "Ndf": {
      "spec": {
        "attributes": {
          "meta": {
            "pair": "USDCNY"
          },
          "tags": [
            "ndf"
          ]
        },
        "base_currency": "CNY",
        "contract_rate": 7.25,
        "domestic_discount_curve_id": "USD-OIS",
        "fixing_date": "2025-03-13",
        "fixing_source_enum": "pboc",
        "forward_rate_override": 7.25,
        "id": "USDCNY-NDF-3M",
        "maturity": "2025-03-15",
        "notional": {
          "amount": "10000000",
          "currency": "CNY"
        },
        "pricing_overrides": {
          "adaptive_bumps": false,
          "call_friction_cents": null,
          "cds_quote_bp": null,
          "credit_spread_bump_bp": null,
          "implied_volatility": null,
          "mc_seed_scenario": null,
          "mean_reversion": null,
          "quoted_clean_price": null,
          "rate_bump_bp": null,
          "rho_bump_decimal": null,
          "spot_bump_pct": null,
          "theta_period": null,
          "tree_steps": null,
          "upfront_payment": null,
          "use_gobet_miri": false,
          "vega_bump_decimal": null,
          "vol_bump_pct": null,
          "vol_surface_extrapolation": "error",
          "ytm_bump_decimal": null
        },
        "quote_convention": "base_per_settlement",
        "settlement_currency": "USD"
      },
      "type": "ndf"
    },
    "QuantoOption": {
      "spec": {
        "attributes": {
          "meta": {},
          "tags": []
        },
        "base_currency": "JPY",
        "correlation": -0.2,
        "day_count": "Act365F",
        "div_yield_id": "NKY-DIV",
        "domestic_discount_curve_id": "USD-OIS",
        "equity_strike": {
          "amount": "35000",
          "currency": "JPY"
        },
        "expiry": "2024-12-20",
        "foreign_discount_curve_id": "JPY-OIS",
        "fx_rate_id": "JPYUSD-SPOT",
        "fx_vol_id": "JPYUSD-VOL",
        "id": "QUANTO-NKY-USD-CALL",
        "notional": {
          "amount": "1000000",
          "currency": "USD"
        },
        "option_type": "call",
        "pricing_overrides": {
          "adaptive_bumps": false,
          "cds_quote_bp": null,
          "implied_volatility": null,
          "mc_seed_scenario": null,
          "quoted_clean_price": null,
          "rate_bump_bp": null,
          "rho_bump_decimal": null,
          "spot_bump_pct": null,
          "theta_period": null,
          "upfront_payment": null,
          "vega_bump_decimal": null,
          "vol_bump_pct": null,
          "ytm_bump_decimal": null
        },
        "quote_currency": "USD",
        "spot_id": "NKY-SPOT",
        "underlying_ticker": "NKY",
        "vol_surface_id": "NKY-VOL"
      },
      "type": "quanto_option"
    }
  },
  "rates": {
    "BasisSwap": {
      "spec": {
        "attributes": {
          "meta": {},
          "tags": []
        },
        "id": "BASIS_SWAP_001",
        "notional": {
          "amount": "10000000",
          "currency": "USD"
        },
        "primary_leg": {
          "bdc": "modified_following",
          "calendar_id": "usny",
          "day_count": "Act360",
          "discount_curve_id": "USD-OIS",
          "end": "2029-01-03",
          "forward_curve_id": "USD-SOFR-3M",
          "frequency": {
            "count": 3,
            "unit": "months"
          },
          "payment_lag_days": 0,
          "reset_lag_days": 0,
          "spread_bp": 5.0,
          "start": "2024-01-03",
          "stub": "ShortFront"
        },
        "reference_leg": {
          "bdc": "modified_following",
          "calendar_id": "usny",
          "day_count": "Act360",
          "discount_curve_id": "USD-OIS",
          "end": "2029-01-03",
          "forward_curve_id": "USD-SOFR-6M",
          "frequency": {
            "count": 6,
            "unit": "months"
          },
          "payment_lag_days": 0,
          "reset_lag_days": 0,
          "spread_bp": 0.0,
          "start": "2024-01-03",
          "stub": "ShortFront"
        }
      },
      "type": "basis_swap"
    },
    "BermudanSwaption": {
      "spec": {
        "bermudan_schedule": {
          "exercise_dates": [
            "2029-01-17",
            "2030-01-17"
          ],
          "lockout_end": null,
          "notice_days": 0
        },
        "bermudan_type": "CoTerminal",
        "day_count": "Thirty360",
        "discount_curve_id": "USD-OIS",
        "fixed_freq": {
          "count": 6,
          "unit": "months"
        },
        "float_freq": {
          "count": 3,
          "unit": "months"
        },
        "forward_curve_id": "USD-OIS",
        "id": "BERM-10NC2-USD",
        "notional": {
          "amount": "10000000",
          "currency": "USD"
        },
        "option_type": "call",
        "settlement": "physical",
        "strike": "0.03",
        "swap_end": "2037-01-17",
        "swap_start": "2027-01-17",
        "vol_surface_id": "USD-SWPNVOL"
      },
      "type": "bermudan_swaption"
    },
    "CallableRangeAccrual": {
      "spec": {
        "attributes": {},
        "call_provision": {
          "call_dates": [
            "2025-07-01"
          ],
          "call_price": 1.0,
          "lockout_periods": 0
        },
        "id": "CALLABLE-RA-PY-E2E",
        "pricing_overrides": {
          "implied_volatility": 1e-12,
          "mc_paths": 8,
          "mean_reversion": 0.05
        },
        "range_accrual": {
          "attributes": {},
          "bounds_type": "absolute",
          "coupon_rate": 0.06,
          "day_count": "Act365F",
          "discount_curve_id": "USD-OIS",
          "div_yield_id": null,
          "id": "RA-PY-E2E",
          "lower_bound": 0.01,
          "notional": {
            "amount": "1000000",
            "currency": "USD"
          },
          "observation_dates": [
            "2025-07-01",
            "2026-01-01",
            "2026-07-01"
          ],
          "past_fixings_in_range": null,
          "payment_date": null,
          "pricing_overrides": {},
          "quanto": null,
          "spot_id": "SOFR-RATE",
          "total_past_observations": null,
          "underlying_ticker": "SOFR",
          "upper_bound": 0.04,
          "vol_surface_id": "SOFR-VOL"
        }
      },
      "type": "callable_range_accrual"
    },
    "CapFloor": {
      "spec": {
        "attributes": {
          "meta": {
            "bloomberg_screen": "SWPM",
            "curve_name": "USD SOFR",
            "curve_number": "490",
            "hw1f_calibration_method": "fixed_kappa_sigma_from_visible_caplet_normal_vol_strip",
            "hw1f_calibration_note": "Bloomberg screenshot shows HW1F and normal-vol VCUB inputs but does not show mean reversion. Kappa is an explicit fixed calibration assumption; sigma is the HW1F short-rate absolute volatility derived from the visible caplet normal-vol strip rather than tuned to expected_outputs.",
            "hw1f_fixed_kappa": "0.0342",
            "hw1f_param_routing": "kappa and sigma are supplied to the cap/floor HW1F tree via the dedicated spec.pricing_overrides.hw1f_mean_reversion and hw1f_sigma fields (the short-rate channel the pricer consumes), not market_quotes.implied_volatility.",
            "hw1f_short_rate_sigma": "0.0094182",
            "hw1f_source_quote": "VCUB USD RFR BVOL Cube visible 5Y tenor caplet normal-vol strip transcribed in inputs.market.surfaces[USD-RFR-BVOL-CUBE-5Y].vols_row_major",
            "hw1f_source_quote_normal_vol": "caplet strip, 20 quarterly expiries",
            "legacy_fixture_name": "usd_cap_5y_atm_black",
            "screen_convexity_adjustment": "Normal",
            "screen_model": "HW1F",
            "screen_volatility_type": "Normal",
            "vol_cube": "USD RFR BVOL Cube"
          },
          "tags": [
            "golden",
            "bloomberg",
            "swpm"
          ]
        },
        "calendar_id": "usny",
        "day_count": "Act360",
        "discount_curve_id": "USD-SOFR-DISC",
        "forward_curve_id": "USD-SOFR",
        "frequency": {
          "count": 3,
          "unit": "months"
        },
        "id": "USD-SOFR-CAP-5Y-ATM-SWPM-20260502",
        "maturity": "2031-05-06",
        "notional": {
          "amount": "10000000",
          "currency": "USD"
        },
        "pricing_overrides": {
          "hw1f_mean_reversion": 0.0342,
          "hw1f_sigma": 0.0094182,
          "tree_steps": 300
        },
        "rate_option_type": "cap",
        "start_date": "2026-05-05",
        "strike": "0.0366561",
        "vol_surface_id": "USD-RFR-BVOL-CUBE-5Y",
        "vol_type": "normal"
      },
      "type": "cap_floor"
    },
    "CmsOption": {
      "spec": {
        "accrual_fractions": [
          0.25,
          0.25,
          0.25,
          0.25
        ],
        "attributes": {
          "meta": {},
          "tags": []
        },
        "cms_tenor": 10.0,
        "day_count": "Act365F",
        "discount_curve_id": "USD-OIS",
        "fixing_dates": [
          "2025-03-20",
          "2025-06-20",
          "2025-09-22",
          "2025-12-22"
        ],
        "forward_curve_id": "USD-LIBOR-3M",
        "id": "CMSOPT-10Y-USD",
        "notional": {
          "amount": "10000000",
          "currency": "USD"
        },
        "option_type": "call",
        "payment_dates": [
          "2025-03-20",
          "2025-06-20",
          "2025-09-22",
          "2025-12-22"
        ],
        "pricing_overrides": {
          "adaptive_bumps": false,
          "cds_quote_bp": null,
          "implied_volatility": null,
          "mc_seed_scenario": null,
          "quoted_clean_price": null,
          "rate_bump_bp": null,
          "rho_bump_decimal": null,
          "spot_bump_pct": null,
          "theta_period": null,
          "upfront_payment": null,
          "vega_bump_decimal": null,
          "vol_bump_pct": null,
          "ytm_bump_decimal": null
        },
        "strike": 0.025,
        "vol_surface_id": "USD-CMS10Y-VOL"
      },
      "type": "cms_option"
    },
    "CmsSpreadOption": {
      "spec": {
        "attributes": {},
        "day_count": "Act365F",
        "discount_curve_id": "USD-OIS",
        "expiry_date": "2026-01-01",
        "forward_curve_id": "USD-SOFR-3M",
        "id": "CMS-SPREAD-PY-E2E",
        "long_cms_tenor": {
          "count": 10,
          "unit": "years"
        },
        "long_vol_surface_id": "USD-SWAPTION-VOL-10Y",
        "notional": {
          "amount": "10000000",
          "currency": "USD"
        },
        "option_type": "call",
        "payment_date": "2026-01-05",
        "pricing_overrides": {},
        "short_cms_tenor": {
          "count": 2,
          "unit": "years"
        },
        "short_vol_surface_id": "USD-SWAPTION-VOL-2Y",
        "spread_correlation": 0.5,
        "strike": 0.005
      },
      "type": "cms_spread_option"
    },
    "CmsSwap": {
      "spec": {
        "attributes": {},
        "cms_accrual_fractions": [
          0.25,
          0.25,
          0.25,
          0.25
        ],
        "cms_day_count": "Act365F",
        "cms_fixing_dates": [
          "2025-03-20",
          "2025-06-20",
          "2025-09-22",
          "2025-12-22"
        ],
        "cms_payment_dates": [
          "2025-06-20",
          "2025-09-22",
          "2025-12-22",
          "2026-03-20"
        ],
        "cms_spread": 0.0,
        "cms_tenor": 10.0,
        "discount_curve_id": "USD-OIS",
        "forward_curve_id": "USD-LIBOR-3M",
        "funding_leg": {
          "accrual_fractions": [
            0.25,
            0.25,
            0.25,
            0.25
          ],
          "day_count": "Thirty360",
          "payment_dates": [
            "2025-06-20",
            "2025-09-22",
            "2025-12-22",
            "2026-03-20"
          ],
          "rate": 0.03,
          "type": "Fixed"
        },
        "id": "CMSSWAP-10Y-USD",
        "notional": {
          "amount": "10000000",
          "currency": "USD"
        },
        "pricing_overrides": {
          "adaptive_bumps": false,
          "call_friction_cents": null,
          "cds_quote_bp": null,
          "credit_spread_bump_bp": null,
          "implied_volatility": null,
          "mc_seed_scenario": null,
          "mean_reversion": null,
          "quoted_clean_price": null,
          "rate_bump_bp": null,
          "rho_bump_decimal": null,
          "spot_bump_pct": null,
          "theta_period": null,
          "tree_steps": null,
          "upfront_payment": null,
          "use_gobet_miri": false,
          "vega_bump_decimal": null,
          "vol_bump_pct": null,
          "vol_surface_extrapolation": "error",
          "ytm_bump_decimal": null
        },
        "side": "pay",
        "swap_convention": "u_s_d_standard",
        "swap_float_day_count": "Act360",
        "vol_surface_id": "USD-CMS10Y-VOL"
      },
      "type": "cms_swap"
    },
    "Deposit": {
      "spec": {
        "attributes": {},
        "day_count": "Act360",
        "discount_curve_id": "USD-OIS",
        "id": "DEP-3M",
        "maturity": "2025-04-15",
        "notional": {
          "amount": 1000000.0,
          "currency": "USD"
        },
        "quote_rate": 0.0505,
        "start_date": "2025-01-15"
      },
      "type": "deposit"
    },
    "ForwardRateAgreement": {
      "spec": {
        "attributes": {},
        "day_count": "Act360",
        "discount_curve_id": "USD-OIS",
        "fixed_rate": "0.048",
        "fixing_date": "2025-04-10",
        "forward_curve_id": "USD-SOFR-3M",
        "id": "FRA-3x6",
        "maturity": "2025-07-15",
        "notional": {
          "amount": "10000000",
          "currency": "USD"
        },
        "pricing_overrides": {},
        "reset_lag": 2,
        "side": "receive",
        "start_date": "2025-04-15"
      },
      "type": "forward_rate_agreement"
    },
    "InflationCapFloor": {
      "spec": {
        "attributes": {},
        "bdc": "modified_following",
        "calendar_id": null,
        "day_count": "Act365F",
        "discount_curve_id": "USD-OIS",
        "frequency": {
          "count": 1,
          "unit": "years"
        },
        "id": "INFLCAP-USD-5Y",
        "inflation_index_id": "US-CPI",
        "lag_override": {
          "months": 3
        },
        "maturity": "2029-01-15",
        "notional": {
          "amount": "1000000",
          "currency": "USD"
        },
        "option_type": "cap",
        "pricing_overrides": {
          "adaptive_bumps": false,
          "call_friction_cents": null,
          "cds_quote_bp": null,
          "credit_spread_bump_bp": null,
          "implied_volatility": null,
          "mc_seed_scenario": null,
          "mean_reversion": null,
          "quoted_clean_price": null,
          "rate_bump_bp": null,
          "rho_bump_decimal": null,
          "spot_bump_pct": null,
          "theta_period": null,
          "tree_steps": null,
          "upfront_payment": null,
          "use_gobet_miri": false,
          "vega_bump_decimal": null,
          "vol_bump_pct": null,
          "vol_surface_extrapolation": "error",
          "ytm_bump_decimal": null
        },
        "start_date": "2024-01-15",
        "strike": "0.03",
        "stub": "ShortFront",
        "vol_surface_id": "USD-INFL-VOL"
      },
      "type": "inflation_cap_floor"
    },
    "InflationSwap": {
      "spec": {
        "attributes": {
          "meta": {},
          "tags": []
        },
        "base_cpi": null,
        "day_count": "Act365F",
        "discount_curve_id": "USD-OIS",
        "fixed_rate": 0.02,
        "id": "INFLSWAP-USD-5Y",
        "inflation_index_id": "US-CPI",
        "lag_override": null,
        "maturity": "2029-01-15",
        "notional": {
          "amount": "1000000",
          "currency": "USD"
        },
        "side": "pay",
        "start_date": "2024-01-15"
      },
      "type": "inflation_swap"
    },
    "InterestRateFuture": {
      "spec": {
        "attributes": {
          "meta": {},
          "tags": []
        },
        "contract_specs": {
          "convexity_adjustment": 0.0,
          "delivery_months": 3,
          "face_value": 1000000.0,
          "tick_size": 0.0025,
          "tick_value": 6.25
        },
        "day_count": "Act360",
        "discount_curve_id": "USD-OIS",
        "expiry": "2025-03-17",
        "fixing_date": "2025-03-17",
        "forward_curve_id": "USD-SOFR-3M",
        "id": "IRF-SR3-MAR25",
        "notional": {
          "amount": "1000000",
          "currency": "USD"
        },
        "period_end": "2025-06-18",
        "period_start": "2025-03-19",
        "position": "long",
        "quoted_price": 95.5,
        "vol_surface_id": null
      },
      "type": "interest_rate_future"
    },
    "InterestRateSwap": {
      "spec": {
        "attributes": {},
        "fixed": {
          "bdc": "modified_following",
          "calendar_id": null,
          "compounding_simple": true,
          "day_count": "Thirty360",
          "discount_curve_id": "USD-OIS",
          "end": "2030-04-15",
          "frequency": {
            "count": 6,
            "unit": "months"
          },
          "par_method": null,
          "rate": 0.045,
          "start": "2025-04-15",
          "stub": "None"
        },
        "float": {
          "bdc": "modified_following",
          "calendar_id": null,
          "compounding": "Simple",
          "day_count": "Act360",
          "discount_curve_id": "USD-OIS",
          "end": "2030-04-15",
          "forward_curve_id": "USD-SOFR-3M",
          "frequency": {
            "count": 3,
            "unit": "months"
          },
          "reset_lag_days": 2,
          "spread_bp": 0.0,
          "start": "2025-04-15",
          "stub": "None"
        },
        "id": "IRS-5Y-USD",
        "notional": {
          "amount": 10000000.0,
          "currency": "USD"
        },
        "side": "pay"
      },
      "type": "interest_rate_swap"
    },
    "IrFutureOption": {
      "spec": {
        "attributes": {},
        "discount_curve_id": "USD-OIS",
        "expiry": "2025-06-16",
        "futures_price": 95.5,
        "id": "IRFO-SOFR-3M-CALL-9550",
        "notional": {
          "amount": "1000000",
          "currency": "USD"
        },
        "option_type": "call",
        "pricing_overrides": {
          "adaptive_bumps": false,
          "call_friction_cents": null,
          "cds_quote_bp": null,
          "credit_spread_bump_bp": null,
          "implied_volatility": null,
          "mc_seed_scenario": null,
          "mean_reversion": null,
          "quoted_clean_price": null,
          "rate_bump_bp": null,
          "rho_bump_decimal": null,
          "spot_bump_pct": null,
          "theta_period": null,
          "tree_steps": null,
          "upfront_payment": null,
          "use_gobet_miri": false,
          "vega_bump_decimal": null,
          "vol_bump_pct": null,
          "vol_surface_extrapolation": "error",
          "ytm_bump_decimal": null
        },
        "strike": 95.5,
        "tick_size": 0.0025,
        "tick_value": 6.25,
        "volatility": 0.2
      },
      "type": "ir_future_option"
    },
    "RangeAccrual": {
      "spec": {
        "attributes": {
          "meta": {},
          "tags": []
        },
        "bounds_type": "relative_to_initial_spot",
        "coupon_rate": 0.08,
        "day_count": "Act365F",
        "discount_curve_id": "USD-OIS",
        "div_yield_id": "SPX-DIV",
        "id": "RANGE-SPX-1Y",
        "lower_bound": 0.95,
        "notional": {
          "amount": "100000",
          "currency": "USD"
        },
        "observation_dates": [
          "2024-01-31",
          "2024-02-29",
          "2024-03-31",
          "2024-04-30",
          "2024-05-31",
          "2024-06-30",
          "2024-07-31",
          "2024-08-31",
          "2024-09-30",
          "2024-10-31",
          "2024-11-30",
          "2024-12-31"
        ],
        "past_fixings_in_range": null,
        "payment_date": null,
        "pricing_overrides": {
          "adaptive_bumps": false,
          "cds_quote_bp": null,
          "implied_volatility": null,
          "mc_seed_scenario": null,
          "quoted_clean_price": null,
          "rate_bump_bp": null,
          "rho_bump_decimal": null,
          "spot_bump_pct": null,
          "theta_period": null,
          "upfront_payment": null,
          "vega_bump_decimal": null,
          "vol_bump_pct": null,
          "ytm_bump_decimal": null
        },
        "spot_id": "SPX-SPOT",
        "total_past_observations": null,
        "underlying_ticker": "SPX",
        "upper_bound": 1.05,
        "vol_surface_id": "SPX-VOL"
      },
      "type": "range_accrual"
    },
    "Repo": {
      "spec": {
        "attributes": {},
        "bdc": "following",
        "calendar_id": "usny",
        "cash_amount": {
          "amount": "10000000",
          "currency": "USD"
        },
        "collateral": {
          "collateral_type": "General",
          "instrument_id": "UST-10Y",
          "market_value_id": "UST_10Y_PRICE",
          "quantity": 103554.0
        },
        "day_count": "Act360",
        "discount_curve_id": "USD-OIS",
        "haircut": 0.02,
        "id": "REPO-GC-7D",
        "maturity": "2025-01-22",
        "pricing_overrides": {},
        "repo_rate": "0.0525",
        "repo_type": "Term",
        "start_date": "2025-01-15",
        "triparty": false
      },
      "type": "repo"
    },
    "Snowball": {
      "spec": {
        "attributes": {},
        "callable": null,
        "coupon_cap": null,
        "coupon_dates": [
          "2025-01-01",
          "2025-07-01",
          "2026-01-01",
          "2026-07-01"
        ],
        "coupon_floor": 0.0,
        "day_count": "Act365F",
        "discount_curve_id": "USD-OIS",
        "fixed_rate": 0.05,
        "floating_index_id": "USD-SOFR-6M",
        "floating_tenor": {
          "count": 6,
          "unit": "months"
        },
        "id": "SNOWBALL-PY-E2E",
        "initial_coupon": 0.03,
        "leverage": 1.0,
        "notional": {
          "amount": "1000000",
          "currency": "USD"
        },
        "pricing_overrides": {
          "implied_volatility": 1e-12,
          "mc_paths": 32,
          "mean_reversion": 0.05
        },
        "variant": "snowball"
      },
      "type": "snowball"
    },
    "Swaption": {
      "spec": {
        "attributes": {},
        "calendar_id": null,
        "cash_settlement_method": "isda_par_par",
        "day_count": "Thirty360",
        "discount_curve_id": "USD-OIS",
        "exercise_style": "european",
        "expiry": "2027-01-15",
        "fixed_freq": {
          "count": 6,
          "unit": "months"
        },
        "float_freq": {
          "count": 3,
          "unit": "months"
        },
        "forward_curve_id": "USD-OIS",
        "id": "SWPN-1Yx5Y-USD",
        "notional": {
          "amount": "10000000",
          "currency": "USD"
        },
        "option_type": "call",
        "pricing_overrides": {
          "adaptive_bumps": false,
          "call_friction_cents": null,
          "cds_quote_bp": null,
          "credit_spread_bump_bp": null,
          "implied_volatility": null,
          "mc_seed_scenario": null,
          "mean_reversion": null,
          "quoted_clean_price": null,
          "rate_bump_bp": null,
          "rho_bump_decimal": null,
          "spot_bump_pct": null,
          "theta_period": null,
          "tree_steps": null,
          "upfront_payment": null,
          "use_gobet_miri": false,
          "vega_bump_decimal": null,
          "vol_bump_pct": null,
          "vol_surface_extrapolation": "error",
          "ytm_bump_decimal": null
        },
        "sabr_params": null,
        "settlement": "cash",
        "strike": "0.03",
        "swap_end": "2032-01-17",
        "swap_start": "2027-01-17",
        "vol_model": "black",
        "vol_surface_id": "USD-SWPNVOL"
      },
      "type": "swaption"
    },
    "Tarn": {
      "spec": {
        "attributes": {},
        "coupon_dates": [
          "2025-01-01",
          "2025-07-01",
          "2026-01-01",
          "2026-07-01"
        ],
        "coupon_floor": 0.0,
        "day_count": "Act365F",
        "discount_curve_id": "USD-OIS",
        "fixed_rate": 0.06,
        "floating_index_id": "USD-SOFR-6M",
        "floating_tenor": {
          "count": 6,
          "unit": "months"
        },
        "id": "TARN-PY-E2E",
        "notional": {
          "amount": "1000000",
          "currency": "USD"
        },
        "pricing_overrides": {
          "implied_volatility": 1e-12,
          "mc_paths": 32,
          "mean_reversion": 0.05
        },
        "target_coupon": 1.0
      },
      "type": "tarn"
    },
    "XccySwap": {
      "spec": {
        "attributes": {
          "meta": {},
          "tags": []
        },
        "id": "XCCY_SWAP_001",
        "leg1": {
          "allow_calendar_fallback": false,
          "bdc": "modified_following",
          "calendar_id": "usny",
          "currency": "USD",
          "day_count": "Act360",
          "discount_curve_id": "USD-OIS",
          "end": "2029-01-03",
          "forward_curve_id": "USD-SOFR-3M",
          "frequency": {
            "count": 3,
            "unit": "months"
          },
          "notional": {
            "amount": "10000000",
            "currency": "USD"
          },
          "payment_lag_days": 0,
          "side": "receive",
          "spread_bp": 0.0,
          "start": "2024-01-03",
          "stub": "ShortFront"
        },
        "leg2": {
          "allow_calendar_fallback": false,
          "bdc": "modified_following",
          "calendar_id": "target2",
          "currency": "EUR",
          "day_count": "Act360",
          "discount_curve_id": "EUR-ESTR",
          "end": "2029-01-03",
          "forward_curve_id": "EUR-EURIBOR-3M",
          "frequency": {
            "count": 3,
            "unit": "months"
          },
          "notional": {
            "amount": "9200000",
            "currency": "EUR"
          },
          "payment_lag_days": 0,
          "side": "pay",
          "spread_bp": 10.0,
          "start": "2024-01-03",
          "stub": "ShortFront"
        },
        "notional_exchange": "initial_and_final",
        "reporting_currency": "USD"
      },
      "type": "xccy_swap"
    },
    "YoYInflationSwap": {
      "spec": {
        "attributes": {},
        "bdc": "modified_following",
        "calendar_id": null,
        "day_count": "Act365F",
        "discount_curve_id": "USD-OIS",
        "fixed_rate": "0.025",
        "frequency": {
          "count": 1,
          "unit": "years"
        },
        "id": "YOYSWAP-USD-5Y",
        "inflation_index_id": "US-CPI",
        "lag_override": {
          "months": 3
        },
        "maturity": "2029-01-15",
        "notional": {
          "amount": "1000000",
          "currency": "USD"
        },
        "pricing_overrides": {
          "adaptive_bumps": false,
          "call_friction_cents": null,
          "cds_quote_bp": null,
          "credit_spread_bump_bp": null,
          "implied_volatility": null,
          "mc_seed_scenario": null,
          "mean_reversion": null,
          "quoted_clean_price": null,
          "rate_bump_bp": null,
          "rho_bump_decimal": null,
          "spot_bump_pct": null,
          "theta_period": null,
          "tree_steps": null,
          "upfront_payment": null,
          "use_gobet_miri": false,
          "vega_bump_decimal": null,
          "vol_bump_pct": null,
          "vol_surface_extrapolation": "error",
          "ytm_bump_decimal": null
        },
        "side": "pay",
        "start_date": "2024-01-15"
      },
      "type": "yoy_inflation_swap"
    }
  }
}
''')

ok = 0
for module_name, instruments in CATALOG.items():
    mod = importlib.import_module(f"finstack_quant.valuations.instruments.{module_name}")
    for class_name, spec in instruments.items():
        cls = getattr(mod, class_name)
        inst = cls.from_json(json.dumps(spec))   # build typed instrument
        inst.validate()                          # structural validation
        assert len(inst.to_json()) > 0            # canonical serialization
        ok += 1
    print(f"{module_name:18} {len(instruments):2d} instrument classes validated")
print(f"\nTotal: {ok} typed instrument classes built + validated + serialized")


commodity           6 instrument classes validated
credit_derivatives  4 instrument classes validated
equity             13 instrument classes validated
exotics             4 instrument classes validated
fixed_income       12 instrument classes validated
fx                 10 instrument classes validated
rates              21 instrument classes validated

Total: 70 typed instrument classes built + validated + serialized


## End-to-end pricing

The pricing pass below derives the market-data IDs referenced by `CATALOG`, builds one synthetic `MarketContext` from typed market-data classes, and then prices every instrument independently. Residual failures are annotated as example scaffolding gaps rather than raised, so the notebook remains runnable while still showing the full typed pricing surface.

In [11]:
from collections import defaultdict

MARKET_ID_MAP = defaultdict(set)
OPERATIONAL_REFERENCE_IDS = defaultdict(set)


def _walk_market_references(value, path=()):
    if isinstance(value, dict):
        for key, item in value.items():
            if isinstance(item, str) and (key.endswith("_id") or key in {"index_id", "ticker"}):
                yield key, item, path + (key,)
            yield from _walk_market_references(item, path + (key,))
    elif isinstance(value, list):
        for idx, item in enumerate(value):
            yield from _walk_market_references(item, path + (str(idx),))


def _market_kind(module_name, field, value):
    if field in {"discount_curve_id", "domestic_discount_curve_id", "foreign_discount_curve_id"}:
        return "discount_curve"
    if field == "credit_curve_id" and module_name == "fixed_income" and value == "USD-CREDIT-BBB":
        # The convertible example labels this credit curve, but its registered pricer requests a discount-style spread curve.
        return "discount_curve"
    if field in {"credit_curve_id", "credit_index_id"}:
        return "hazard_curve" if field == "credit_curve_id" else "credit_index"
    if field in {"inflation_index_id"}:
        return "inflation_curve"
    if field in {"vol_index_curve_id"} or value == "VIX":
        return "vol_index_curve"
    if field.endswith("vol_surface_id") or field in {"vol_surface_id", "fx_vol_id", "vol_of_vol_surface_id"}:
        return "vol_surface"
    if field in {"forward_curve_id", "floating_index_id", "leg1_forward_curve_id", "leg2_forward_curve_id"}:
        return "price_curve" if module_name == "commodity" or value.endswith("SPOT-AVG") else "forward_curve"
    if field == "index_id":
        if value.startswith("SOFR") or value.startswith("USD-") or value.startswith("EUR-"):
            return "forward_curve"
        return "scalar_price"
    if field in {"spot_id", "price_id", "market_value_id", "fx_spot_id", "fx_rate_id", "yield_id", "duration_id", "convexity_id", "bond_id", "ctd_bond_id"}:
        return "scalar_price"
    if field == "div_yield_id":
        return "dividend_yield"
    return "operational_reference"


for module_name, instruments in CATALOG.items():
    for class_name, payload in instruments.items():
        for field, value, _path in _walk_market_references(payload["spec"]):
            kind = _market_kind(module_name, field, value)
            if kind == "operational_reference":
                OPERATIONAL_REFERENCE_IDS[kind].add(value)
            else:
                MARKET_ID_MAP[kind].add(value)

# A few pricing engines request companion market objects that are implied by the spec rather than named directly.
MARKET_ID_MAP["discount_curve"].update({"EUR-OIS", "JPY-OIS", "USD-TREASURY"})
MARKET_ID_MAP["price_curve"].update({"CL-FORWARD", "WTI-FORWARD", "NG-FORWARD", "RBOB-FORWARD", "NG-SPOT-AVG"})
MARKET_ID_MAP["vol_surface"].update({"JPYUSD-VOL", "RBOB-VOL", "TECH-VOL", "USD-SWAPTION-VOL-2Y", "USD-SWAPTION-VOL-10Y"})
MARKET_ID_MAP["sabr_cube"].update({"USD-RFR-BVOL-CUBE-5Y", "USD-SWPNVOL", "USD-CMS10Y-VOL", "USD-SWAPTION-VOL-2Y", "USD-SWAPTION-VOL-10Y", "VIX-VOLVOL"})
MARKET_ID_MAP["scalar_price"].update({"CNYUSD-SPOT", "EURUSD-SPOT", "JPYUSD-SPOT", "TECH", "UST-10Y", "US91282CJL54"})

print("Derived market references:")
for kind in sorted(MARKET_ID_MAP):
    print(f"  {kind:16} {len(MARKET_ID_MAP[kind]):2d} ids")
print("Operational ids annotated if required:", len(OPERATIONAL_REFERENCE_IDS["operational_reference"]))


Derived market references:
  credit_index      1 ids
  discount_curve   10 ids
  dividend_yield    5 ids
  forward_curve     8 ids
  hazard_curve      2 ids
  inflation_curve   1 ids
  price_curve       5 ids
  sabr_cube         6 ids
  scalar_price     18 ids
  vol_index_curve   1 ids
  vol_surface      19 ids
Operational ids annotated if required: 23


In [12]:
from datetime import date, timedelta

from finstack_quant.core.market_data import (
    DiscountCurve,
    ForwardCurve,
    FxMatrix,
    HazardCurve,
    InflationCurve,
    MarketContext,
    PriceCurve,
    VolCube,
    VolSurface,
    VolatilityIndexCurve,
)

AS_OF = date(2024, 1, 15)

DISCOUNT_KNOTS = [
    (0.0, 1.0000),
    (0.5, 0.9802),
    (1.0, 0.9608),
    (2.0, 0.9231),
    (5.0, 0.8187),
    (10.0, 0.6703),
    (30.0, 0.3010),
]
FORWARD_KNOTS = [
    (0.0, 0.0420),
    (0.5, 0.0430),
    (1.0, 0.0435),
    (2.0, 0.0440),
    (5.0, 0.0450),
    (10.0, 0.0460),
    (30.0, 0.0470),
]
PRICE_CURVE_LEVELS = {
    "CL-FORWARD": 76.0,
    "WTI-FORWARD": 74.0,
    "NG-FORWARD": 3.20,
    "RBOB-FORWARD": 2.40,
    "NG-SPOT-AVG": 3.20,
}
SCALAR_PRICES = {
    "AAPL-SPOT": 185.0,
    "CNYUSD-SPOT": 0.1400,
    "EQUITY-SPOT": 100.0,
    "EURUSD": 1.08,
    "EURUSD-SPOT": 1.08,
    "JPYUSD-SPOT": 0.0068,
    "NKY-SPOT": 39000.0,
    "SOFR-RATE": 0.045,
    "SPX-SPOT": 5200.0,
    "TECH": 42.0,
    "US-CORP-CONVEXITY": 0.02,
    "US-CORP-DURATION": 7.0,
    "US-CORP-INDEX": 100.0,
    "US-CORP-YIELD": 0.052,
    "US91282CJL54": 99.5,
    "UST-10Y": 99.5,
    "UST10Y-PRICE": 99.5,
    "UST_10Y_PRICE": 99.5,
    "VIX": 18.0,
}
DIVIDEND_YIELDS = {
    "AAPL-DIV": 0.006,
    "EQUITY-DIVYIELD": 0.010,
    "NKY-DIV": 0.012,
    "SPX-DIV": 0.015,
    "SPX-DIVYIELD": 0.015,
}
FORWARD_TENORS = {
    "EUR-EURIBOR-3M": 0.25,
    "SOFR-1M": 1.0 / 12.0,
    "SOFR-3M": 0.25,
    "USD-LIBOR-3M": 0.25,
    "USD-SOFR": 0.25,
    "USD-SOFR-1M": 1.0 / 12.0,
    "USD-SOFR-3M": 0.25,
    "USD-SOFR-6M": 0.50,
}


def _price_curve_knots(level):
    return [
        (0.0, level),
        (0.5, level * 1.01),
        (1.0, level * 1.02),
        (2.0, level * 1.03),
        (5.0, level * 1.05),
    ]


def _is_rate_vol_surface(surface_id):
    return any(token in surface_id for token in ("SWPN", "SWAPTION", "SOFR", "CMS", "RFR", "INFL"))


def _fixing_series(series_id, as_of):
    observations = []
    current = date(2023, 1, 1)
    end = as_of + timedelta(days=400)
    while current <= end:
        observations.append([current.isoformat(), 0.043])
        current += timedelta(days=1)
    return {
        "id": series_id,
        "currency": None,
        "observations": observations,
        "interpolation": "step",
    }


def build_shared_market_json(as_of=AS_OF):
    market = MarketContext()

    discount_ids = set(MARKET_ID_MAP["discount_curve"])
    forward_ids = set(MARKET_ID_MAP["forward_curve"]) - discount_ids - set(MARKET_ID_MAP["price_curve"])
    price_curve_ids = set(MARKET_ID_MAP["price_curve"])
    hazard_ids = set(MARKET_ID_MAP["hazard_curve"]) - discount_ids

    for curve_id in sorted(discount_ids):
        market.insert(DiscountCurve(curve_id, as_of, DISCOUNT_KNOTS, day_count="act_365f"))

    for curve_id in sorted(forward_ids):
        tenor = FORWARD_TENORS.get(curve_id, 0.25)
        market.insert(ForwardCurve(curve_id, tenor, FORWARD_KNOTS, base_date=as_of, day_count="act_365f"))

    for curve_id in sorted(price_curve_ids):
        level = PRICE_CURVE_LEVELS.get(curve_id, 100.0)
        market.insert(PriceCurve(curve_id, as_of, _price_curve_knots(level), day_count="act_365f"))

    for curve_id in sorted(hazard_ids):
        market.insert(
            HazardCurve(
                curve_id,
                as_of,
                [(0.25, 0.015), (1.0, 0.018), (3.0, 0.022), (5.0, 0.026), (10.0, 0.032)],
                recovery_rate=0.40,
                day_count="act_365f",
            )
        )

    for curve_id in sorted(MARKET_ID_MAP["inflation_curve"]):
        market.insert(
            InflationCurve(
                curve_id,
                as_of,
                315.0,
                [(0.0, 315.0), (1.0, 322.9), (2.0, 331.0), (5.0, 357.2), (10.0, 405.0)],
                day_count="act_365f",
            )
        )

    expiries = [0.01, 0.10, 0.25, 0.50, 1.0, 2.0, 5.0, 10.0]
    strikes = [0.0, 0.01, 0.03, 0.05, 0.08, 1.0, 20.0, 50.0, 75.0, 100.0, 150.0, 250.0, 4500.0, 5200.0, 6000.0, 40000.0]
    for surface_id in sorted(MARKET_ID_MAP["vol_surface"]):
        quote_type = "normal" if _is_rate_vol_surface(surface_id) else "black_lognormal"
        market.insert(
            VolSurface(
                surface_id,
                expiries,
                strikes,
                [0.25] * (len(expiries) * len(strikes)),
                secondary_axis="strike",
                interpolation_mode="vol",
                quote_type=quote_type,
            )
        )

    cube_expiries = [0.10, 0.25, 0.50, 1.0, 2.0, 5.0, 10.0]
    cube_tenors = [0.25, 1.0, 2.0, 5.0, 10.0, 30.0]
    cube_node_count = len(cube_expiries) * len(cube_tenors)
    cube_params = [{"alpha": 0.035, "beta": 0.50, "rho": -0.20, "nu": 0.40, "shift": 0.02}] * cube_node_count
    for cube_id in sorted(MARKET_ID_MAP["sabr_cube"]):
        market.insert(VolCube(cube_id, cube_expiries, cube_tenors, cube_params, [0.045] * cube_node_count))

    for curve_id in sorted(MARKET_ID_MAP["vol_index_curve"]):
        market.insert(VolatilityIndexCurve(curve_id, as_of, [(0.0, 18.0), (0.25, 19.0), (1.0, 20.0), (2.0, 21.0)]))

    fx = FxMatrix()
    fx.set_quote("EUR", "USD", 1.08)
    fx.set_quote("JPY", "USD", 0.0068)
    fx.set_quote("CNY", "USD", 0.1400)
    market.insert_fx(fx)

    for price_id in sorted(MARKET_ID_MAP["scalar_price"] | set(SCALAR_PRICES)):
        value = SCALAR_PRICES.get(price_id, 100.0)
        currency = None if price_id in {"CNYUSD-SPOT", "EURUSD", "EURUSD-SPOT", "JPYUSD-SPOT", "SOFR-RATE", "VIX", "US-CORP-CONVEXITY", "US-CORP-DURATION", "US-CORP-YIELD"} else "USD"
        market.insert_price(price_id, value, currency)
    for div_id in sorted(MARKET_ID_MAP["dividend_yield"] | set(DIVIDEND_YIELDS)):
        market.insert_price(div_id, DIVIDEND_YIELDS.get(div_id, 0.01), None)

    state = json.loads(market.to_json())
    state["fx"] = {
        "config": {"pivot_currency": "USD", "enable_triangulation": True, "cache_capacity": 256},
        "quotes": [["EUR", "USD", 1.08], ["JPY", "USD", 0.0068], ["CNY", "USD", 0.1400]],
    }

    # Some JSON-only pricers read scalar quotes from a compact price map.
    state.setdefault("prices", {})
    for price_id, value in SCALAR_PRICES.items():
        if price_id in {"CNYUSD-SPOT", "EURUSD", "EURUSD-SPOT", "JPYUSD-SPOT", "SOFR-RATE", "VIX", "US-CORP-CONVEXITY", "US-CORP-DURATION", "US-CORP-YIELD"}:
            state["prices"][price_id] = {"unitless": value}
        else:
            state["prices"][price_id] = {"price": {"amount": value, "currency": "USD"}}
    for div_id, value in DIVIDEND_YIELDS.items():
        state["prices"][div_id] = {"unitless": value}

    state["curves"].append(
        {
            "type": "base_correlation",
            "id": "CDX-BC",
            "detachment_points": [3.0, 7.0, 10.0, 15.0, 30.0, 100.0],
            "correlations": [0.30, 0.30, 0.30, 0.30, 0.30, 0.30],
        }
    )
    state["credit_indices"] = [
        {
            "id": "CDX.NA.IG.HAZARD",
            "num_constituents": 125,
            "recovery_rate": 0.40,
            "index_credit_curve_id": "CDX.NA.IG.HAZARD",
            "base_correlation_curve_id": "CDX-BC",
        }
    ]
    state["series"] = [
        _fixing_series("FIXING:USD-SOFR-3M", as_of),
        _fixing_series("FIXING:USD-SOFR-6M", as_of),
        _fixing_series("FIXING:SOFR-3M", as_of),
        _fixing_series("FIXING:SOFR-1M", as_of),
        _fixing_series("FIXING:NG-SPOT-AVG", as_of),
    ]
    return json.dumps(state)


SHARED_MARKET_JSON = build_shared_market_json()
SHARED_MARKET = json.loads(SHARED_MARKET_JSON)
print(
    "Shared market:",
    f"{len(SHARED_MARKET['curves'])} curves,",
    f"{len(SHARED_MARKET['surfaces'])} surfaces,",
    f"{len(SHARED_MARKET['vol_cubes'])} SABR cubes,",
    f"{len(SHARED_MARKET['prices'])} scalar prices",
)

Shared market: 27 curves, 19 surfaces, 6 SABR cubes, 24 scalar prices


In [13]:
from collections import Counter

from finstack_quant.valuations import ValuationResult

# Default models price the broad majority once the shared market is complete. Keep this map
# intentionally small; add entries only when a registered non-default model materially helps.
MODEL_OVERRIDES = {}

RESIDUAL_NOTES = {
    "bermudan_swaption": "needs calibrated Hull-White params or a pre-calibrated tree exposed through the binding",
}


def _as_valuation_result(price_output):
    if isinstance(price_output, str):
        return ValuationResult.from_json(price_output)
    return price_output


def _short_error(exc):
    message = str(exc).split("\n")[0]
    return message.replace("Validation error: ", "")[:180]


def _failure_note(type_tag, exc):
    if type_tag in RESIDUAL_NOTES:
        return RESIDUAL_NOTES[type_tag]
    return "needs: " + _short_error(exc)


def _price_text(value):
    try:
        return f"{float(value):,.2f}"
    except (TypeError, ValueError):
        return str(value)


PRICE_ROWS = []
RESIDUAL_ROWS = []

for module_name, instruments in CATALOG.items():
    module = importlib.import_module(f"finstack_quant.valuations.instruments.{module_name}")
    for class_name, payload in instruments.items():
        cls = getattr(module, class_name)
        inst = cls.from_json(json.dumps(payload))
        inst.validate()
        type_tag = payload["type"]
        try:
            if type_tag in MODEL_OVERRIDES:
                price_output = inst.price(SHARED_MARKET_JSON, AS_OF.isoformat(), model=MODEL_OVERRIDES[type_tag])
            else:
                price_output = inst.price(SHARED_MARKET_JSON, AS_OF.isoformat())
            result = _as_valuation_result(price_output)
            PRICE_ROWS.append(
                {
                    "module": module_name,
                    "class": class_name,
                    "type": type_tag,
                    "price": getattr(result, "price", None),
                    "currency": str(getattr(result, "currency", "")),
                }
            )
        except TypeError as exc:
            # Credit-derivative wrappers expose a no-model signature; other wrappers accept the default keyword.
            try:
                price_output = inst.price(SHARED_MARKET_JSON, AS_OF.isoformat(), model="default")
                result = _as_valuation_result(price_output)
                PRICE_ROWS.append(
                    {
                        "module": module_name,
                        "class": class_name,
                        "type": type_tag,
                        "price": getattr(result, "price", None),
                        "currency": str(getattr(result, "currency", "")),
                    }
                )
            except Exception as fallback_exc:
                RESIDUAL_ROWS.append(
                    {
                        "module": module_name,
                        "class": class_name,
                        "type": type_tag,
                        "note": _failure_note(type_tag, fallback_exc),
                        "error": _short_error(fallback_exc),
                    }
                )
        except Exception as exc:
            RESIDUAL_ROWS.append(
                {
                    "module": module_name,
                    "class": class_name,
                    "type": type_tag,
                    "note": _failure_note(type_tag, exc),
                    "error": _short_error(exc),
                }
            )

module_totals = Counter()
for module_name, instruments in CATALOG.items():
    module_totals[module_name] = len(instruments)
module_priced = Counter(row["module"] for row in PRICE_ROWS)
module_residual = Counter(row["module"] for row in RESIDUAL_ROWS)

print(f"Priced {len(PRICE_ROWS)}/{sum(module_totals.values())} instruments against one shared MarketContext")
print("\nPer-module summary:")
for module_name in sorted(module_totals):
    print(
        f"  {module_name:18} priced {module_priced[module_name]:2d}/{module_totals[module_name]:2d}; "
        f"annotated {module_residual[module_name]:2d}"
    )

print("\nPriced PVs:")
for row in PRICE_ROWS:
    print(f"  {row['module']:18} {row['class']:28} {_price_text(row['price']):>14} {row['currency']}")

print("\nAnnotated residuals:")
for row in RESIDUAL_ROWS:
    print(f"  {row['module']:18} {row['class']:28} {row['note']} ({row['error']})")

Priced 63/70 instruments against one shared MarketContext

Per-module summary:
  commodity          priced  6/ 6; annotated  0
  credit_derivatives priced  4/ 4; annotated  0
  equity             priced 13/13; annotated  0
  exotics            priced  4/ 4; annotated  0
  fixed_income       priced  9/12; annotated  3
  fx                 priced  9/10; annotated  1
  rates              priced 18/21; annotated  3

Priced PVs:
  commodity          CommodityAsianOption               9,091.25 USD
  commodity          CommodityForward                       0.00 USD
  commodity          CommodityOption                    8,802.77 USD
  commodity          CommoditySpreadOption                  0.00 USD
  commodity          CommoditySwap                    -25,016.26 USD
  commodity          CommoditySwaption                 33,978.46 USD
  credit_derivatives CDSIndex                         399,031.29 USD
  credit_derivatives CDSOption                        384,959.05 USD
  credit_derivatives

## Takeaways

- All instrument classes share `from_json` / `validate` / `to_json` / `price`; the JSON discriminator `"type"` selects the class.
- The typed surface is the object-oriented counterpart to the JSON-first `price_instrument` / `price_instrument_with_metrics` functions used in the per-asset-class notebooks.
- Credit-derivative classes (`CDSIndex`, `CDSOption`, `CDSTranche`, `CreditDefaultSwap`) return a `ValuationResult` object directly from `price` (no `model=` argument).